In [1]:
import os 
os.environ['OPENAI_API_KEY']='sk-proj-'



In [2]:
from openai import OpenAI
import os
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
import lance

import pandas as pd
import pyarrow as pa
import pyarrow.dataset
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")  



In [21]:
df = pd.read_csv('spotify_records.csv')

In [22]:
df.columns = df.columns.str.replace('.', '_')
df.columns = df.columns.str.replace(' ', '_')

In [335]:
# import torch

# print(torch.cuda.is_available())  # Should return True if CUDA is working
# print(torch.version.cuda) 

In [9]:
dd = model.encode(df["Content"])

In [29]:
df['embedding'] = dd.tolist()

In [37]:
# df

In [14]:
# !rm -rf tmp/test.parquet

In [50]:
# df = pd.DataFrame({"a": [5], "b": [10]})
table_uri = 'spotify2'
uri = "tmp/"+table_uri+".parquet"
lance_uri = "tmp/"+table_uri+".lance"
# tbl = pa.Table.from_pandas(df)
# pa.dataset.write_dataset(tbl, uri, format='parquet')

# parquet = pa.dataset.dataset(uri, format='parquet')
# lance.write_dataset(parquet, "tmp/spotify1.lance")

In [ ]:
embedding_field = pa.field("embedding", pa.list_(pa.float32(), 384))  
new_schema = tbl.schema.append(embedding_field)
df['embedding'] = dd.tolist()
table = pa.Table.from_pandas(df, schema=new_schema)
pa.dataset.write_dataset(table, uri, format="parquet")


In [ ]:
parquet = pa.dataset.dataset(uri, format='parquet')
lance.write_dataset(parquet, lance_uri)

In [52]:
import duckdb
import lancedb
dataset = lance.dataset(lance_uri)


In [53]:
dataset.create_index("embedding",
                    index_type="IVF_PQ",
                    num_partitions=256,  # IVF
                    num_sub_vectors=96)

In [56]:
lance.write_dataset(dataset, 'tmp/indexed.lance')

In [332]:
# dataset.lance_schema

In [333]:
# df

In [334]:
# d.filter['_distance']

In [ ]:
# dataset = lance.dataset

In [57]:
duckdb.query("SELECT * FROM dataset LIMIT 1").to_df()

,ID,Source,CreatedAt,Language,Record_Sentiment,Tracked_Keywords,Reasons,Content,Summary,metadata_AppID,...,metadata_Score,metadata_Timestamp,metadata_Useful,metadata_Author,metadata_CountryISO2,metadata_CreatedAt,metadata_Rating,metadata_Upvotes,metadata_Version,embedding
0,41a33eca-3cea-55bc-9eff-96cc5d8a881f,Playstore,2025-03-12T04:39:31Z,spa,NaN,Advertisements,"Happy With The Service Of Spotify, Happy With ...",Excellent especially when there are no ads,"The user praised Spotify for being excellent, ...",com.spotify.music,...,5.0,1.741754e+09,0.0,None,None,NaN,NaN,NaN,9.0.24.601,"[-0.060554452, -0.0347185, -0.027859842, 0.013..."


In [380]:
import re
def get_schema_dump() -> str:
    # Use duckdb to retrieve the first 2 rows as a schema dump
    head_df = duckdb.query("SELECT * FROM dataset LIMIT 3").to_df()
    del head_df['embedding']
    return head_df.to_json(orient="records")


def clean_json_output(json_str: str) -> str:
    """
    Remove markdown code fences (``` or ```json) from the JSON string if present.
    """
    json_str = json_str.strip()
    if json_str.startswith("```"):
        # Remove starting and ending code fences using regex
        pattern = r"^```(?:json)?\s*(.*?)\s*```$"
        match = re.search(pattern, json_str, flags=re.DOTALL)
        if match:
            return match.group(1).strip()
        else:
            # Fallback: remove first and last lines if they are fences
            lines = json_str.splitlines()
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines = lines[:-1]
            return "\n".join(lines).strip()
    return json_str

def generate_and_execute_sql_queries(natural_query: str) -> str:
    schema_dump = get_schema_dump()
    client = OpenAI()
    prompt = f"""
You are a SQL design expert. Given the following schema dump from dataset.head(2):
{schema_dump}

Create a JSON array of executable SQL queries to analyze the data based on this natural language query: "{natural_query}".
Do not provide any explanation, always include limit 50 to the sql if limit is not specified by the query, just output the JSON array of SQL queries as a string.
    """
    try:
        completion = client.chat.completions.create(
           model="gpt-4o",
           messages=[
             {"role": "system", "content": "You and a SQL design expert create complex queries."},
             {"role": "user", "content": prompt}
           ]
        )
        queries_str = completion.choices[0].message.content.strip()
        queries_str = clean_json_output(queries_str)
        queries = json.loads(queries_str)
        # print(queries)
    except Exception as gen_error:
        # If JSON parsing or LLM call fails, ask LLM for a correction with error details.
        error_prompt = f"""
You are a SQL design expert. There was an error generating the SQL queries: {str(gen_error)}.
Given the following schema dump from dataset.head(2):
{schema_dump}

and the natural language query: "{natural_query}",
please return a valid JSON array of executable SQL queries as a string.
        """
        correction_completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You and a SQL design expert create complex queries."},
                {"role": "user", "content": error_prompt}
            ]
        )
        queries_str = correction_completion.choices[0].message.content.strip()
        queries_str = clean_json_output(queries_str)
        queries = json.loads(queries_str)
    
    sql_results = []
    for query in queries:
        result_data = None
        current_query = query
        try:
            result_df = duckdb.query(current_query).to_df()
            result_data = result_df.to_dict(orient="records")
        except Exception as exec_error:
            # Attempt to correct the individual query using LLM, with the error message provided.
            error_prompt = f"""
You are a SQL design expert. The following SQL query produced an error: {str(exec_error)}.
Original query: {current_query}
Please return a corrected, executable SQL query as a plain string.
            """
            correction_completion = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You and a SQL design expert create complex queries."},
                    {"role": "user", "content": error_prompt}
                ]
            )
            corrected_query = correction_completion.choices[0].message.content.strip()
            corrected_query = clean_json_output(corrected_query)
            try:
                result_df = duckdb.query(corrected_query).to_df()
                result_data = result_df.to_dict(orient="records")
                current_query = corrected_query  # Update to corrected query.
            except Exception as exec_error2:
                result_data = {"error": str(exec_error2)}
        sql_results.append({"sql": current_query, "output": result_data})
    
    output = {
        "original_query": natural_query,
        "sql_created": sql_results
    }
    if len(sql_results)==0:
        output = ''
    return json.dumps(output)

sql_query_agent = Agent( name= 'QuatitativeDatasetQueryAgent',
              role="Gives analysis on top of data recieved from tools",
              instructions=[
                    '''give natural query modified to a list of language queries max 5 (ask to quote some examples as well) that are focused 
                  towards quantitative analysis as an input to tools call one by one to gather all the data for analysis.
                  prepare the final analysis based on the outputs keeping in mind the actual query. 
                  Dont include any unconclusive finding, or any error.
                  If no relavant information from tool is recieved just tell that query is not relavant for this dataset.
                  '''
                ],
              tools=[generate_and_execute_sql_queries], 
              show_tool_calls=True, 
              markdown=True
             )

# sql_query_agent = Agent(tools=[generate_and_execute_sql_queries], show_tool_calls=True, markdown=True)


In [301]:
sql_query_agent.print_response("what are customers issues about ads?", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about ads?                                                                            ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • generate_and_execute_sql_queries(natural_query=Summarize the top complaints received from customers about     ┃
┃ advertisements.)                                                                                                ┃
┃ • generate_and_execute_sql_queries(natural_query=List the number of customer complaints related to advertising  ┃
┃ per month over the last year.)                                                                                  ┃
┃ • generate_and_execute_sql_queries(natural_query=Identify if there is any trend in the customer complaints      ┃
┃ about ads.)                                                                                                     ┃
┃ • generate_and_execute_sql_queries(natural_query=Quote some examples of customer complaints related to          ┃
┃ advertisements.)                                                                                                ┃
┃ • generate_and_execute_sql_queries(natural_query=Calculate the average resolution time for                      ┃
┃ advertisement-related complaints.)                                                                              ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (37.8s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                      Analysis of Customer Issues about Ads                                      ┃
┃                                                                                                                 ┃
┃ Based on the queries executed, here is a comprehensive analysis regarding customer complaints about             ┃
┃ advertisements:                                                                                                 ┃
┃                                                                                                                 ┃
┃  1 Top Complaints about Advertisements:                                                                         ┃
┃     • Excessive Ads: This is a major concern, with customers frequently reporting excessive ads on the free     ┃
┃       version of platforms like Spotify.                                                                        ┃
┃     • Frequent Ads on Spotify: Many users have issues with the frequency of ads interrupting their music on     ┃
┃       Spotify.                                                                                                  ┃
┃     • Unskippable Ads: Complaints about ads that cannot be skipped are also prevalent.                          ┃
┃     • Ads on Paid Subscriptions: Some customers report receiving ads even though they have paid subscriptions.  ┃
┃  2 Examples of Customer Complaints: Here are some summarized customer reviews that highlight the issues:        ┃
┃     • A user notes excessive ads e

In [295]:
sql_query_agent.print_response("what are customers issues about playlist? quote some examples.", stream=True)

▰▰▰▰▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist? quote some examples.                                                  ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • generate_and_execute_sql_queries(natural_query=Count the number of customer issues related to playlists.)     ┃
┃ • generate_and_execute_sql_queries(natural_query=List the top 5 most common customer issues about playlists.)   ┃
┃ • generate_and_execute_sql_queries(natural_query=Extract quotes or examples of customer feedback mentioning     ┃
┃ playlist issues.)                                                                                               ┃
┃ • generate_and_execute_sql_queries(natural_query=Analyze the sentiment of customer feedback specifically about  ┃
┃ playlists.)                                                                                                     ┃
┃ • generate_and_execute_sql_queries(natural_query=Identify any patterns or trends in the timing of               ┃
┃ playlist-related issues reported by customers.)                                                                 ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (51.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                   Analysis of Customer Issues about Playlists                                   ┃
┃                                                                                                                 ┃
┃ After analyzing the data collected regarding customer issues related to playlists, the following insights were  ┃
┃ gathered:                                                                                                       ┃
┃                                                                                                                 ┃
┃ 1. Volume of Playlist-Related Issues:                                                                           ┃
┃                                                                                                                 ┃
┃  • There are 4,207 documented issues specifically related to playlists.                                         ┃
┃                                                                                                                 ┃
┃ 2. Common Issues:                                                                                               ┃
┃                                                                                                                 ┃
┃  • Shuffle Mode Problems: Customers frequently express dissatisfaction with Spotify's shuffle play feature.     ┃
┃    Many users are frustrated by the 'smart shuffle' automatically playing unintended music rather than their    ┃
┃    playlists. This feature often results in repetitive playback instead of true randomization.                  ┃
┃  • Advertisement Concerns: Many users find the advertisement timing excessive, sometimes interrupting playlists ┃
┃    and leading to dissatisfaction,

In [303]:
sql_query_agent.print_response("what is moon?", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what is moon?                                                                                                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (3.0s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ The query regarding "what is moon?" is unlikely to be relevant for quantitative analysis using datasets         ┃
┃ typically accessible through SQL databases. This query is more about general knowledge or scientific            ┃
┃ information, and less about data-driven insights that can be queried from a database. If you have a specific    ┃
┃ question about the moon that relates to data or requires quantitative analysis, feel free to adjust your query  ┃
┃ accordingly.                                                                                                    ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [375]:
# from typing import Optional, List
def embedding_based_query(natural_query: str) -> str:
    """
    Query the dataset using embeddings and return a list of strings from the 'Content' column.
    Assumes that 'dataset' and 'model' are already defined and initialized.
    """
    d = dataset.to_table(
        nearest={"column": "embedding", "k": 500, "q": model.encode(natural_query)},
        filter="_distance < 1.5"
    )
    content = [a for a in d['Content'].to_pylist()]
    return json.dumps(content)

embedding_query_agent = Agent(
    name='QualitativeDatasetQueryAgent',
    role="Gives detailed analysis on top of data received from tools",
    instructions=[
        '''Give a natural query modified into a list (maximum 3) of language queries that are focused 
towards qualitative analysis. Quote some examples if needed. These queries will be used as input 
to tools, called one by one to gather all data for analysis. Then, prepare the final analysis based 
on the outputs, keeping in mind the original query. Do not include any unconclusive findings or errors.
If no relevant information is received from the tool, simply state that the query is not relevant for this dataset.'''
    ],
    tools=[embedding_based_query],
    show_tool_calls=True,
    markdown=True
)

In [344]:
embedding_query_agent.print_response("what are customers issues about playlist?", stream=True)

▰▰▰▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist?                                                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • embedding_based_query(natural_query=What problems do customers report about the playlist feature?)            ┃
┃ • embedding_based_query(natural_query=How do customers describe their experience with managing playlists?)      ┃
┃ • embedding_based_query(natural_query=What feedback do customers give on playlist usability?)                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (21.1s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                           Analysis of Customer Issues Regarding the Playlist Feature:                           ┃
┃                                                                                                                 ┃
┃  1 Usability Concerns:                                                                                          ┃
┃     • Complexity in Creation and Organization: Some customers find the process of organizing playlists to       ┃
┃       require many steps and suggest the UI could be simpler (e.g., "Organising playlist requires many steps,   ┃
┃       UI could have been simple"). This can make it particularly difficult for users who are on mobile devices  ┃
┃       or have limited technical skills.                                                                         ┃
┃     • Lack of Intuitive Features: There are reports of confusion between playlists and liked lists, and         ┃
┃       difficulty finding playlists (e.g., "Confusing what goes to playlists and what goes to liked lists and    ┃
┃       how to transfer from one to the other?", "Most difficult user interface.").                               ┃
┃     • Complaints about Auto Shuffle: The auto shuffle feature is often criticized for being ineffective ("Auto  ┃
┃       shuffle is terrible. Spotify cache grows significantly large over time, making it more demanding on       ┃
┃       system resources").                                                                                       ┃
┃  2 Relevance and Content of Playlists:                                                                          ┃
┃     • Mismatch in Recommended Songs: A significant issue is the autoplay of recommended songs that do not align ┃
┃       with user tastes after a playlist ends ("The principle of creating playlists is to listen to sounds that  ┃
┃       we like, but 5 songs of 3 minutes that do not interest me and that follow one another while I should be   ┃
┃       listening to sounds that I HAVE selected...").                                                            ┃
┃     • AI-Generated Content: Some users express dissatisfaction with the AI-generated playlists, suggesting they ┃
┃       lack genuine customization or accuracy ("Every playlist is AI nothing genuine", "They are only using AI   ┃
┃       generated suggested playlist

In [457]:
embedding_query_agent.print_response("content quality issues", stream=True)

▰▰▰▰▰▰▰ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ content quality issues                                                                                          ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • embedding_based_query(natural_query=What are common complaints about content quality?)                        ┃
┃ • embedding_based_query(natural_query=What suggestions are being made to improve the quality of content?)       ┃
┃ • embedding_based_query(natural_query=What positive feedback exists about content quality?)                     ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (8.7s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Here's a qualitative analysis based on the dataset, addressing content quality issues:                          ┃
┃                                                                                                                 ┃
┃                                     Common Complaints about Content Quality                                     ┃
┃                                                                                                                 ┃
┃  1 Over-promotion and Advertising: Users have expressed frustration over excessive ads and frequent             ┃
┃    self-promotion. This includes complaints about constant requests for reviews and advertising that disrupts   ┃
┃    the user experience.                                                                                         ┃
┃     • Example: "Stop constantly asking for reviews and ratings."                                                ┃
┃     • Example: "Too much advertising and you have to pay to have peace."                                        ┃
┃  2 Content Variety and Availability: There is discontent regarding limited freedom of expression and content    ┃
┃    availability that is geographically restricted.                                                              ┃
┃     • Example: "Very little freedom of expression on this platform."                                            ┃
┃     • Example: "Certain features are geographically unavailable, I believe the world is a global village, we    ┃
┃       should equally have the same features around the world."                                                  ┃
┃  3 Lack of User-Centric Features: Annoyances include unprofessional services, poor suggestion algorithms, and   ┃
┃    features that fail to meet user expectations.                                                                ┃
┃     • Example: "A bunch of unprofessionals, neither funny nor smart."                                           ┃
┃     • Example: "The suggestions are pathetic."                                                                  ┃
┃                                                                                                                 ┃
┃                                    Suggestions for Improving Content Quality                                    ┃
┃                                   

In [345]:
embedding_query_agent.print_response("what is moon?", stream=True)

▰▰▰▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what is moon?                                                                                                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • embedding_based_query(natural_query=What is the significance of the Moon in cultural contexts?)               ┃
┃ • embedding_based_query(natural_query=Describe the Moon's role in scientific research and exploration.)         ┃
┃ • embedding_based_query(natural_query=How does the Moon influence Earth?)                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (11.0s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ The query regarding the Moon appears to be unrelated to the current dataset. No specific insights or relevant   ┃
┃ information about the Moon's significance in cultural contexts, role in scientific research, or influence on    ┃
┃ Earth have been identified in the dataset.                                                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [350]:
# import importlib
# importlib.invalidate_caches()
# import hirag


In [464]:
import re
import openai
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.csv import CSVKnowledgeBase
# from agno.vectordb.pgvector import PgVector
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.team import Team
from IPython.display import Markdown, display
# ---------------------------
# Set up CSV Knowledge Base (Internal Knowledge)
# ---------------------------
# vector_db = PgVector(
#     table_name="csv_documents",
#     db_url="postgresql+psycopg://ai:ai@localhost:5532/ai",  # Update credentials as needed
# )
# csv_kb = CSVKnowledgeBase(
#     path="path/to/your/data.csv",  # Update this path to your CSV file
#     vector_db=vector_db,
# )
# csv_kb.load(recreate=False)

# ---------------------------
# Specialized Agents for the Research Team
# ---------------------------
query_refiner_agent = Agent(
    name="QueryRefinerAgent",
    role="Refine and simplify the research query. by analyzing it from historical, current, and future perspectives.",
    model=OpenAIChat(id="o3-mini"),
    instructions=['''Plan the given query from research perspective thinking about the past, present and future. 
                  Give the list of questions that need answering. stick to input query refinement task only'''],
    show_tool_calls=False,
    markdown=True,
)

# class InternalResearchAgent(Agent):
#     def probe(self, query):
#         result = self.run(query).content.strip()
#         return len(result) > 50  # Returns True if content is substantial

# internal_research_agent = InternalResearchAgent(
#     name="InternalResearchAgent",
#     role="Search the CSV-based internal knowledge for relevant information.",
#     model=OpenAIChat(id="gpt-4o"),
#     knowledge=csv_kb,
#     instructions=["Retrieve relevant internal information from the CSV knowledge base."],
#     show_tool_calls=False,
#     markdown=True,
# )

external_research_agent = Agent(
    name="ExternalResearchAgent",
    role="Fetch up-to-date information from the web.",
    model=OpenAIChat(id="gpt-4o"),
    tools=[DuckDuckGoTools()],
    instructions=["Perform a web search and include sources in your response."],
    show_tool_calls=True,
    markdown=True,
)
external_research_agent2 = Agent(
    name="ExternalResearchAgentOpenai",
    role="Fetch up-to-date information from the web.",
    model=OpenAIChat(id="gpt-4o-search-preview"),
    instructions=["Perform a web search and include sources in your response and technical details if present"],
    show_tool_calls=True,
    markdown=True,
)

summarization_agent = Agent(
    name="SummarizationAgent",
    role="Synthesize the research inputs into a coherent report.",
    model=OpenAIChat(id="o3-mini"),
    instructions=["A final deep research report with clear, well-organized insights and verified facts and technical details if present."],
    reasoning=True,
    show_tool_calls=False,
    markdown=True,
)

    
citation_agent = Agent(
    name="CitationAgent",
    role="Extract and format citations from text.",
    model=OpenAIChat(id="gpt-4o-mini-search-preview"),
    instructions=["Extract cited sources, URLs, or references from the text."],
    show_tool_calls=False,
    markdown=True,
)

# trend_agent = Agent(
#     name="TrendAnalysisAgent",
#     role="Identify recent trends related to the query.",
#     model=OpenAIChat(id="gpt-4o-mini-search-preview"),
#     instructions=["Identify any recent trends or developments relevant to the query."],
#     show_tool_calls=False,
#     markdown=True,
# )

# ---------------------------
# Specialized Agents for the Synthesis Team
# ---------------------------
# synthesis_agent = Agent(
#     name="SynthesisAgent",
#     role="Synthesize a comprehensive research report from provided inputs.",
#     model=OpenAIChat(id="gpt-4o"),
#     instructions=[
#         "Using the following inputs, generate a detailed and well-organized research report.",
#         "Inputs: Original Query, Refined Query, Internal Summary, External Summary, Trends, and Citations."
#     ],
#     show_tool_calls=False,
#     markdown=True,
# )

# fact_checker_agent = Agent(
#     name="FactCheckerAgent",
#     role="Verify the factual accuracy of the report.",
#     model=OpenAIChat(id="gpt-4o-mini-search-preview"),
#     instructions=["Check the report for factual errors and inconsistencies."],
#     show_tool_calls=False,
#     markdown=True,
# )

# critique_agent = Agent(
#     name="CritiqueAgent",
#     role="Critique the final report and suggest improvements.",
#     model=OpenAIChat(id="gpt-4o-mini-search-preview"),
#     instructions=["Provide constructive feedback and improvement suggestions."],
#     show_tool_calls=False,
#     markdown=True,
# )

# ---------------------------
# Decision Agent with Probe Approach
# ---------------------------
# class DecisionAgentWithProbe:
#     """
#     Determines which sub-agents to invoke based on the query and a probe
#     of the internal knowledge base.
#     """
#     def __init__(self):
#     # def __init__(self, internal_agent):
#         # self.internal_agent = internal_agent
#         self.agent = Agent(
#             name="DecisionAgent",
#             role="Decide which sub-agents are needed based on the query and internal knowledge relevance.",
#             model=OpenAIChat(id="gpt-4o"),
#             instructions=[
#                 "You are a strategic decision-maker for a multi-agent research system.",
#                 "Given the research query and a hint about internal knowledge relevance, decide which of the following agents to invoke:",
#                 "InternalResearchAgent, ExternalResearchAgent, QueryRefinerAgent, SummarizationAgent,",
#                 "CitationAgent, TrendAnalysisAgent, FactCheckerAgent, CritiqueAgent.",
#                 "Return a comma-separated list of agent names in lowercase."
#             ],
#             show_tool_calls=False,
#             markdown=True,
#         )
    
#     def decide(self, query):
#         # internal_relevant = self.internal_agent.probe(query)
#         # hint = "internal relevant" if internal_relevant else "internal not relevant"
#         # decision_prompt = f"{query}\nHint: {hint}"
#         decision_prompt = f"{query}"
#         response = self.agent.run(decision_prompt)
#         decision_text = response.content.strip()
#         decisions = [agent.strip().lower() for agent in decision_text.split(',')]
#         return decisions

# decision_agent = DecisionAgentWithProbe(internal_research_agent)
decision_agent = DecisionAgentWithProbe()

# ---------------------------
# Improved Tournament Evaluator Agent
# ---------------------------
# class TournamentEvaluatorAgent:
#     """
#     Uses an LLM prompt to evaluate multiple candidate responses and selects the best one.
#     """
#     def evaluate(self, candidates, category, instructions=None, max_tokens=100):
#         if instructions is None:
#             instructions = (
#                 f"Rank the following candidate responses for '{category}' by quality, clarity, and relevance. "
#                 "Return only the candidate number (1-indexed) of the best candidate."
#             )
#         prompt = instructions + "\n\n"
#         for idx, cand in enumerate(candidates):
#             prompt += f"Candidate {idx+1}:\n{cand}\n\n"
#         prompt += "Which candidate is best? Output just the candidate number."
#         response = openai.ChatCompletion.create(
#             model="gpt-4o",
#             messages=[
#                 {"role": "system", "content": "You are an impartial evaluator."},
#                 {"role": "user", "content": prompt}
#             ],
#             max_tokens=max_tokens
#         )
#         eval_result = response["choices"][0]["message"]["content"].strip()
#         try:
#             best_index = int(eval_result) - 1
#             if best_index < 0 or best_index >= len(candidates):
#                 best_index = 0
#         except Exception:
#             best_index = 0
#         return candidates[best_index]

# ---------------------------
# Define Agent Teams
# ---------------------------
# Research Team: Gathers and processes input.
research_team = Team(
    mode="coordinate",
    members=[
        # query_refiner_agent,
        # internal_research_agent,
        # external_research_agent,
        sql_query_agent,
        embedding_query_agent,
        external_research_agent2,
        query_refiner_agent,
        summarization_agent,
        citation_agent,
        # trend_agent,
    ],
    model=OpenAIChat(id="o3-mini"),
    success_criteria="Comprehensive and organized research summaries with relevant citations and trend analysis.",
    instructions=['''
    1. **Data Gathering from Internal Sources**  
       - Start by querying the `embedding_query_agent`. If its output indicates that the query is relevant to our internal dataset, gather detailed information next by invoking the `sql_query_agent`.  
       - Collect and consolidate all relevant findings from these agents, ensuring they directly address the original query.
       - If Query is relavant to Internal Sources, then tell in detail the hirearchy and instances for each hirearchy for the report along with examples. You should be able to cover all the themes and not just the top themes. .

    2. **Fallback to External Research**  
       - If the `embedding_query_agent` returns no relevant information, refine the query using the `query_refiner_agent`. Then, provide both the refined query and the original query one by one to the `external_research_agent2` for web-based research.  
       - You may invoke `external_research_agent2` multiple times (up to 3) to handle subqueries. Collect all the findings and links, then invoke next step.

    3. **Citations and Final Summarization**  
       - Whenever external information is used, invoke the `citation_agent` to include proper citations or external links.  
       - Finally, pass all accumulated knowledge and context to the `summarization_agent` to produce a well-structured, detailed summary that remains focused on the original query.

    Ensure that each step only includes relevant points and excludes any inconclusive or erroneous information.
'''],
    show_tool_calls=True,
    # share_member_interactions=True,
    enable_agentic_context=True,
    markdown=True,
    debug_mode=False,
    show_members_responses=True,
    # enable_team_history=True,
    num_of_interactions_from_history=5,
)


In [395]:
query = "what are customers issues about playlist?"

In [390]:
# research_team.print_response(json.dumps({'query':query,'refined_query':refined_query}), stream=True)
# out = research_team.run(query).content

In [429]:
research_team.print_response("what are customers issues about playlist?", stream=True)

▰▰▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist?                                                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (57.3s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is the detailed summary report based on recent customer feedback regarding issues with playlists:         ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃  # Customer Playlist Experience Report                                                                          ┃
┃                                                                                                                 ┃
┃  ## Introduction                                                                                                ┃
┃                                                                                                                 ┃
┃  Recent customer feedback highlights key issues impacting the user experience with playlist management on our   ┃
┃  platform. Users are increasingly frustrated by unauthorized playlist alterations, disruptive ads and           ┃
┃  recommendations, and limitations imposed on free accounts. This report synthesizes detailed feedback from      ┃
┃  multiple customers to pinpoint the underlying factors driving dissatisfaction and outlines recommendations fo  ┃
┃  improvements.                                                                                                  ┃
┃                                                                                                                 ┃
┃  ## Detailed Findings                                                                                           ┃
┃                                                                                                                 ┃
┃  ### 1. Playlist Management Issues                                                                              ┃
┃                                                                                                                 ┃
┃  Customers report that their playlists are frequently altered without their consent. Specifically, users        ┃
┃  indicate that the system introduces random songs into their carefully curated playlists. This unexpected       ┃
┃  behavior not only disrupts the listening experience but also pushes users toward upgrading their accounts for  ┃
┃  supposedly better experience.                                                                                  ┃
┃                                                                                                                 ┃
┃  - **Customer Feedback:**                                                                                       ┃
┃    > "It adds random songs to my playlist and I have to pay to play a single song."                             ┃
┃                                                                                                                 ┃
┃  ### 2. Disruptions from Ads and Forced Recommendations                                                         ┃
┃                                                                                                                 ┃
┃  Another significant concern is th

In [436]:
research_team.print_response("what are customers issues about playlist?", stream=True)

▰▰▰▰▰▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist?                                                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (35.9s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a hierarchical breakdown of customer issues about playlists based on our internal dataset. The         ┃
┃ findings were gathered using our embedding-based query from the internal qualitative feedback, and they         ┃
┃ organize the reported issues into clear categories along with representative instances/examples for each:       ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                1. Persistent Ads and Premium Access Requirements                                ┃
┃                                                                                                                 ┃
┃  • Hierarchy Name: Ad Interruptions & Premium Barriers                                                          ┃
┃    Instances/Details:                                                                                           ┃
┃     • Excessive Advertisements:                                                                                 ┃
┃       • Users express frustration over sudden ad interruptions interrupting their listening experience,         ┃
┃       particularly for free or non-premium users.                                                               ┃
┃     • Premium Upgrades Mandate:                                                                                 ┃
┃       • Many complaints indicate that customers feel forced to upgrade to a premium (paid) account to enjoy     ┃
┃       uninterrupted, exactly curated playlists.                                                                 ┃
┃     • Example Instance:                                                                                         ┃
┃       • A customer mentioned that they “cannot play their complete playlist without enduring frequent ad breaks ┃
┃       or being forced into a shuffled mode.”                                                                    ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                2. Playlist Management and Song Selection Issues                                 ┃
┃                                                                                                                 ┃
┃  • Hierarchy Name: Playlist Organization & Content Management                                                   ┃
┃    Instances/Details:                                                                                           ┃
┃     • Inability to Reorder or Edit:                                                                             ┃
┃       • Users report difficulties in reordering songs within a playlist or editing playlist details, leading to ┃
┃       a less personalized experience.                                                                           ┃
┃     • Unwanted Song Additions:    

In [440]:
research_team.print_response("what are customers issues about playlist?", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist?                                                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (34.8s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Based on analysis from our internal dataset, here are the comprehensive issues customers face regarding         ┃
┃ playlists:                                                                                                      ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                  1. Playlist Control and Recommendation Issues                                  ┃
┃                                                                                                                 ┃
┃  • Unexpected Song Recommendations:                                                                             ┃
┃    Customers have reported that when they select a playlist, the system sometimes plays recommended songs       ┃
┃    instead of the chosen tracks. For example, one user noted,                                                   ┃
┃    "I wanted to complain because when I put on a playlist instead of playing a song that I selected, it plays   ┃
┃    recommended songs, I don't like that."                                                                       ┃
┃    This points to problems with the recommendation algorithm overriding manual selection.                       ┃
┃  • Mismatch with User Taste:                                                                                    ┃
┃    Even when recommendations are offered, they often don’t align with users’ musical preferences, leading to a  ┃
┃    less enjoyable listening experience.                                                                         ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                 2. User Interface (UI) and Experience Concerns                                  ┃
┃                                                                                                                 ┃
┃  • Complex Playlist Management:                                                                                 ┃
┃    Many customers find creating and managing playlists cumbersome. They have indicated that the interface is    ┃
┃    not intuitive enough for tasks such as:                                                                      ┃
┃     • Rearranging songs manually.                                                                               ┃
┃     • Organizing playlists into nested folders for better management.                                           ┃
┃    This complexity detracts from the overall ease-of-use expected by customers.                                 ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                   

In [397]:
research_team.print_response("what are customers issues about playlist?", stream=True)

▰▰▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about playlist?                                                                       ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (6.2s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a consolidated summary of the main customer issues related to playlists:                               ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                               Key Customer Issues                                               ┃
┃                                                                                                                 ┃
┃  1 Shuffle & Song Order Issues:                                                                                 ┃
┃     • Randomization Problems: Users are frustrated with the shuffle feature because it doesn't randomize well,  ┃
┃       often repeating the same songs.                                                                           ┃
┃     • Order Flexibility: Customers cannot easily select and play songs in their preferred order unless they     ┃
┃       subscribe to premium.                                                                                     ┃
┃  2 Playlist Modifications & Disappearance:                                                                      ┃
┃     • Unexpected Changes: There are reports of playlists being modified automatically or even disappearing.     ┃
┃       Sometimes songs are unexpectedly added or removed without user consent.                                   ┃
┃  3 Ads and Premium Feature Restrictions:                                                                        ┃
┃     • Ad Interruptions: The free version suffers from frequent and lengthy ads, detracting from the overall     ┃
┃       listening experience.                                                                                     ┃
┃     • Feature Limitations: Non-premium accounts lack features such as specific song play and download           ┃
┃       capabilities, which pushes users toward the premium subscription.                                         ┃
┃  4 Technical & Performance Issues:                                                                              ┃
┃     • Glitches and Bugs: Users have experienced performance issues, including slow loading, glitches, and       ┃
┃       bugs—especially after app updates.                                                                        ┃
┃     • Resource Consumption: Concerns about the app using excessive data and memory also affect user experience. ┃
┃  5 Content Restriction & Availability:                                                                          ┃
┃     • Regional Limitations: Some songs or albums are not available in certain regions or due to content         ┃
┃       restrictions.                                                                                             ┃
┃     • Download Limitations: Users are restricted from downloading individual tracks and sometimes have to       ┃
┃       download an entire playlist or album, which may not be convenient.                                        ┃
┃                                   

In [441]:
research_team.print_response("what are customers issues about ads?", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about ads?                                                                            ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (33.0s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a detailed summary of customer issues related to ads, organized hierarchically along with examples:    ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                      Hierarchical Categorization of Issues                                      ┃
┃                                                                                                                 ┃
┃ 1. Minor Categories:                                                                                            ┃
┃                                                                                                                 ┃
┃  • Frequent and Prolonged Ads:                                                                                  ┃
┃    Customers are disturbed by the continuous, prolonged display of ads which interrupts their experience.       ┃
┃  • Obnoxious Amount of Ads:                                                                                     ┃
┃    Users feel overwhelmed when ads are displayed in excessive numbers.                                          ┃
┃  • Abuse of Ads:                                                                                                ┃
┃    Some customers report aggressive or intrusive ad practices that imply a misuse of advertising parameters.    ┃
┃  • Irrelevant or Misleading Ads:                                                                                ┃
┃    Ads that do not suit user interests or are perceived as deceptive create frustration.                        ┃
┃  • Excessive Advertising Interruptions:                                                                         ┃
┃    Frequent breaks for ads, particularly when they are unskippable, greatly disrupt the user experience.        ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                         Subcategories or Nuanced Issues                                         ┃
┃                                                                                                                 ┃
┃  • Excessive Number of Ads:                                                                                     ┃
┃    Too many ads displayed at once can lead to a cluttered and unpleasant browsing or viewing experience.        ┃
┃  • Repetitive Ads:                                                                                              ┃
┃    Seeing the same ad repeatedly is a common complaint, with users noticing that the repetition quickly becomes ┃
┃    irritating.                                                                                                  ┃
┃  • Ads Disrupting User Experience:                                                                              ┃
┃    Interruptions that break the fl

In [430]:
research_team.print_response("what are customers issues about ads?", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are customers issues about ads?                                                                            ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (84.4s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is the comprehensive report based on qualitative insights drawn from our internal dataset:                ┃
┃                                                                                                                 ┃
┃ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ┃
┃ ┃                              Customer Issues with Ads: A Comprehensive Report                               ┃ ┃
┃ ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛ ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                                  Introduction                                                   ┃
┃                                                                                                                 ┃
┃ This report synthesizes customer feedback regarding their experiences with ads. It highlights three critical    ┃
┃ areas of concern:                                                                                               ┃
┃                                                                                                                 ┃
┃  1 Intrusiveness and frequency of ads                                                                           ┃
┃  2 Relevancy and targeting issues                                                                               ┃
┃  3 Audio quality and content issues                                                                             ┃
┃                                                                                                                 ┃
┃ The goal is to outline these concerns and provide actionable recommendations for improvement.                   ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                      1. Intrusiveness and Frequency of Ads                                      ┃
┃                                                                                                                 ┃
┃                                                   Key Issues                                                    ┃
┃                                                                                                                 ┃
┃  • Overwhelming Volume: Customers frequently mention that the high volume of ads disrupts their experience.     ┃
┃  • Interruptive Placement: Ads appear at unexpected times, breaking the flow of content.                        ┃
┃  • Repetitiveness: Users express frustration over seeing the same ads repeatedly.                               ┃
┃                                                                                                                 ┃
┃                                   

In [444]:
research_team.print_response("give me some suggested feature requests from data.", stream=True)

▰▰▰▰▰▰▰ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ give me some suggested feature requests from data.                                                              ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (53.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a structured summary of the feature request themes extracted from the internal dataset, along with     ┃
┃ examples for each theme:                                                                                        ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                           1. Performance Enhancements                                           ┃
┃                                                                                                                 ┃
┃  • Speed and Accessibility                                                                                      ┃
┃     • Example: "Search fasting" – Indicates a need for faster search operations.                                ┃
┃     • Example: "Find titles very quickly" – Suggests enhancing the speed of title searches.                     ┃
┃  • Content Availability                                                                                         ┃
┃     • Example: "A lot of content" – Requests for a broader content library.                                     ┃
┃     • Example: "More official content need the be added" – Emphasizes the inclusion of more high-quality,       ┃
┃       official content.                                                                                         ┃
┃  • Data Usage Optimization                                                                                      ┃
┃     • Example: "It's interesting, but usage of DATA NEEDS TO BE MINIMISED" – Highlights a desire to reduce data ┃
┃       consumption.                                                                                              ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                               2. UI Improvements                                                ┃
┃                                                                                                                 ┃
┃  • Ease of Navigation                                                                                           ┃
┃     • Example: "A more easily consultable index would be appreciated. Thank you." – Suggests improving the      ┃
┃       content indexing system.                                                                                  ┃
┃     • Example: "Easy to access, everything is organized and classified" – Commends current organization but     ┃
┃       indicates potential for even more intuitive design.                                                       ┃
┃  • Search and Recommendation Tweaks                                                                             ┃
┃     • Example: "The suggested searches are disturbing.. please remove them." – Requests refinement or removal   ┃
┃       of the suggested search feat

In [400]:
research_team.print_response("give me some suggested feature requests.", stream=True)

▰▰▰▰▰▰▰ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ give me some suggested feature requests.                                                                        ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (7.2s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below are some suggested feature requests based on the customer feedback gathered on playlists and ads:         ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                        Playlist-Related Feature Requests                                        ┃
┃                                                                                                                 ┃
┃  1 Enhanced Shuffle Functionality:                                                                              ┃
┃     • Improve the randomness algorithm to ensure a more varied shuffle experience.                              ┃
┃     • Offer users the ability to adjust or customize shuffle settings based on personal preferences.            ┃
┃  2 Customizable Song Order for Non-Premium Users:                                                               ┃
┃     • Allow non-premium users greater control over song order in playlists, such as manually rearranging tracks ┃
┃       without requiring a subscription.                                                                         ┃
┃  3 Playlist Stability and Management:                                                                           ┃
┃     • Introduce a feature to "lock" playlists to avoid unintended modifications or removals.                    ┃
┃     • Add a history or backup feature so users can restore previous versions if issues occur.                   ┃
┃     • Improve organizational tools, like enabling nested playlists or categorizing playlists for easier         ┃
┃       navigation.                                                                                               ┃
┃  4 Personalized Recommendations and Management:                                                                 ┃
┃     • Enhance recommendation algorithms to provide better personalized suggestions that reflect individual      ┃
┃       musical tastes.                                                                                           ┃
┃     • Allow feedback mechanisms so users can refine recommendations and auto-curated playlists.                 ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                           Ad-Related Feature Requests                                           ┃
┃                                                                                                                 ┃
┃  1 Reduced Ad Interruption:                                                                                     ┃
┃     • Implement a feature that reduces the frequency of ads in the free version without compromising the        ┃
┃       platform's revenue model.                                                                                 ┃
┃     • Introduce shorter ad segment

In [450]:
research_team.print_response("qoute examples of political messages from data.", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ qoute examples of political messages from data.                                                                 ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (28.1s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is an organized summary of the political messages extracted from our internal dataset. In addition to     ┃
┃ simply listing quotes, we have identified a hierarchy of political message themes along with the instances      ┃
┃ (quotes) that illustrate each category. This detailed grouping not only shows the top themes but also covers    ┃
┃ the range of political commentary found in the dataset.                                                         ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────                                                                                   ┃
┃ Hierarchy of Political Message Themes                                                                           ┃
┃                                                                                                                 ┃
┃  1 Criticism of Political Figures and Their Supporters                                                          ┃
┃    These messages express negative opinions or strong disapproval toward political leaders and their            ┃
┃    supporters. They often include derogatory language and express dissatisfaction with the policies or persona  ┃
┃    of a figure.                                                                                                 ┃
┃     • Example 1:                                                                                                ┃
┃       "Idiots. They support trump!"                                                                             ┃
┃       (Here the speaker harshly criticizes a group of Trump supporters.)                                        ┃
┃     • Example 2:                                                                                                ┃
┃       "Application of fascists (supporters of Trump and Musk) I'm leaving!"                                     ┃
┃       (This statement not only dismisses these groups with strong labels but also signals a personal            ┃
┃       disassociation.)                                                                                          ┃
┃  2 Activism, Resistance, and Advocacy                                                                           ┃
┃    In this category, messages highlight solidarity with political causes, champion resistance against perceived ┃
┃    injustices, or promote policy agendas.                                                                       ┃
┃     • Example 1:                                                                                                ┃
┃       "We are lucky to have such a committed group to dig up the dirt on Trump’s actions. I learn something     ┃
┃       important every day and it energizes my actions to resist the attack on our freedoms."                    ┃
┃       (This quote shows pride in activism and commitment to thwart perceived political threats.)                ┃
┃     • Example 2:                                                                                                ┃
┃       "Please support Taiwan Indep

In [451]:
research_team.print_response("qoute issues with artist from data.", stream=True)

▰▰▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ qoute issues with artist from data.                                                                             ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (31.8s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a detailed breakdown of the issues related to artists as obtained from our internal dataset. The       ┃
┃ findings have been organized into hierarchical themes, with specific instances provided as examples for each    ┃
┃ category to ensure comprehensive coverage:                                                                      ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ 1. Payment and Compensation Concerns                                                                            ┃
┃ Many entries highlight worries about inadequate or unfair payment practices for artists. These include direct   ┃
┃ calls to “pay the artists” as well as criticisms about how platforms compensate creative work.                  ┃
┃                                                                                                                 ┃
┃  • Instances and Examples:                                                                                      ┃
┃     • "Abusive price and poorly paid artist"                                                                    ┃
┃     • "Pay artists"                                                                                             ┃
┃     • "Pay your artists"                                                                                        ┃
┃     • "Pay the artists more!"                                                                                   ┃
┃     • "Start paying artist!!!!"                                                                                 ┃
┃     • "Artists deserve the money but not in my budget these days."                                              ┃
┃     • "Make sure that the authors/creators get paid for their work!!!! Today it is incredibly poorly paid to    ┃
┃       create music."                                                                                            ┃
┃     • "This new layout sucks and you dont pay artists nearly enough which is BS because we supported you the    ┃
┃       longest"                                                                                                  ┃
┃     • "Pay as it should be to the artists."                                                                     ┃
┃                                                                                                                 ┃
┃ These entries suggest a strong sentiment among users that the current financial arrangements or compensation    ┃
┃ policies are not meeting the expectations or needs of artists.                                                  ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ 2. Technical/Operational Issues Concerning Artist Visibility                                                    ┃
┃ Some feedback focuses on technical

In [458]:
research_team.print_response("song quality issues", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ song quality issues                                                                                             ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (38.0s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is the detailed report derived from our internal data query on "song quality issues," organized           ┃
┃ hierarchically and supported by specific examples and themes.                                                   ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                          1. General Quality Complaints                                          ┃
┃                                                                                                                 ┃
┃ Hierarchy Details:                                                                                              ┃
┃ At the top level of the data, the general sentiment centers on overall dissatisfaction with the audio fidelity  ┃
┃ of songs. This theme is broad and encompasses several sub-issues.                                               ┃
┃                                                                                                                 ┃
┃ Key Instances & Examples:                                                                                       ┃
┃                                                                                                                 ┃
┃  • Perceived Loss in Quality:                                                                                   ┃
┃     • Users have commented that some songs sound similar to low-quality MP3 files, suggesting concerns over     ┃
┃       excessive file compression.                                                                               ┃
┃     • For example, one comment stated, “Some tracks sound like they're compressed too much, making the music    ┃
┃       feel flat and lifeless.”                                                                                  ┃
┃  • Expectation vs. Reality:                                                                                     ┃
┃     • The mismatch between the presumed quality from high-end platforms and the actual experience on the        ┃
┃       platform was noted, where sound quality is a primary factor in choosing a music service.                  ┃
┃     • As one user mentioned, “The sound quality here is really disappointing compared to what I've heard        ┃
┃       elsewhere.”                                                                                               ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                       2. Comparison with Other Platforms                                        ┃
┃                                                                                                                 ┃
┃ Hierarchy Details:                                                                                              ┃
┃ This theme involves direct compari

In [402]:
research_team.print_response("what is graphrag?", stream=True)

▰▰▰▰▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what is graphrag?                                                                                               ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (27.5s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a detailed explanation of what GraphRAG is:                                                            ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                                What is GraphRAG?                                                ┃
┃                                                                                                                 ┃
┃ GraphRAG is an advanced method within the family of Retrieval-Augmented Generation (RAG) techniques that        ┃
┃ integrates knowledge graphs into the generation process of large language models (LLMs). By using structured    ┃
┃ knowledge graphs, GraphRAG enhances the reasoning capabilities of LLMs, allowing them to produce more accurate  ┃
┃ and context-aware outputs when handling complex queries and datasets.                                           ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                           Key Components of GraphRAG                                            ┃
┃                                                                                                                 ┃
┃  1 Retrieval-Augmented Generation (RAG) Foundation:                                                             ┃
┃     • Traditional RAG methods boost LLM outputs by retrieving relevant contextual information from large text   ┃
┃       corpora. However, such methods often rely on unstructured text snippets which sometimes limit the model's ┃
┃       ability to understand complex relationships.                                                              ┃
┃  2 Integration of Knowledge Graphs:                                                                             ┃
┃     • Structured Representation: GraphRAG extracts entities, relationships, and key claims from unstructured    ┃
┃       text to create a hierarchical and structured knowledge graph.                                             ┃
┃     • Hierarchical Clustering: Using algorithms like the Leiden algorithm, it organizes these entities into     ┃
┃       meaningful communities, thus providing a structured overview of the data.                                 ┃
┃     • Enhanced Retrieval: At query time, the LLM leverages the structured graph to retrieve and synthesize      ┃
┃       information more effectively, leading to improved reasoning and connection of disparate data points.      ┃
┃  3 Query Modes:                                                                                                 ┃
┃     • GraphRAG supports various query modes—such as Global Search, Local Search, and DRIFT Search—each tailored ┃
┃       to efficiently retrieve the most relevant information from the knowledge graph based on the query         ┃
┃       context.                    

In [403]:
research_team.print_response("what is agno?", stream=True)

▰▱▱▱▱▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what is agno?                                                                                                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (20.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a comprehensive explanation of "Agno," which can refer to several distinct concepts across different   ┃
┃ domains:                                                                                                        ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                                1. Agno Framework                                                ┃
┃                                                                                                                 ┃
┃  • Definition:                                                                                                  ┃
┃    An open-source platform designed for building, deploying, and monitoring AI agents. It is tailored to work   ┃
┃    with various large language models (LLMs) and provides functionalities such as memory integration, tool      ┃
┃    extensibility, and scalability.                                                                              ┃
┃  • Context & Applications:                                                                                      ┃
┃     • Facilitates the creation of domain-specific AI agents (e.g., compliance analysts, data analysts).         ┃
┃     • Supports developers in constructing sophisticated AI systems tailored for specific business functions.    ┃
┃  • Reference:                                                                                                   ┃
┃    ]8;id=208678;https://www.aitoolnet.com/pure-ai-agents?utm_source=openai\Agno Framework on AIToolNet]8;;\                                                                                  ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                        2. Agno, Pangasinan (Philippines)                                        ┃
┃                                                                                                                 ┃
┃  • Definition:                                                                                                  ┃
┃    A municipality in the province of Pangasinan, Philippines, named after the indigenous "Agno Castor" tree     ┃
┃    known for its medicinal properties.                                                                          ┃
┃  • Context & Applications:                                                                                      ┃
┃     • Rich historical background, with establishment dating back to 1791.                                       ┃
┃     • Known for natural attractions like the Umbrella Rocks of Sabangan and celebrated cultural events such as  ┃
┃       the annual Umbrella Rocks Festival.                                                                       ┃
┃  • Reference:                                                        

In [404]:
research_team.print_response("what are different types of agents in llm ?", stream=True)

▰▰▰▰▰▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what are different types of agents in llm ?                                                                     ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (32.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below are several common types of agents built using large language models (LLMs), each designed to tackle      ┃
┃ different kinds of tasks or improve reasoning and interaction:                                                  ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                       1. Conversational Agents (Chatbots)                                       ┃
┃                                                                                                                 ┃
┃  • Purpose: Engage in natural language dialogue and answer user queries interactively.                          ┃
┃  • Examples: Virtual assistants, customer support bots.                                                         ┃
┃  • Characteristics: Emphasize fluent conversation, context retention, and user engagement.                      ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                             2. Task-Oriented Agents                                             ┃
┃                                                                                                                 ┃
┃  • Purpose: Complete specific tasks such as appointment scheduling, email drafting, or data extraction.         ┃
┃  • Examples: Personal assistants, scheduling bots.                                                              ┃
┃  • Characteristics: Often incorporate structured flows and integrate with external APIs or calendars to execute ┃
┃    tasks.                                                                                                       ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                              3. Tool-Using Agents                                               ┃
┃                                                                                                                 ┃
┃  • Purpose: Extend the LLM’s capabilities by interacting with external tools and APIs.                          ┃
┃  • Examples: Python execution agents, web scraping bots.                                                        ┃
┃  • Characteristics: Designed based on frameworks like ReAct, these agents generate and execute code, leverage   ┃
┃    databases, or use retrieval systems to enhance responses.                                                    ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                   

In [413]:
research_team.print_response("give me some suggested feature requests.", stream=True)

▰▰▰▰▰▰▰ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ give me some suggested feature requests.                                                                        ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (29.8s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Based on our internal dataset, here are some suggested feature requests collected from user feedback:           ┃
┃                                                                                                                 ┃
┃  1 Reduce Advertisements / Ad-Free Option                                                                       ┃
┃     • Several users mentioned that the frequency of ads disrupts their music experience.                        ┃
┃     • Proposals include fewer ads overall or offering an ad-free version without requiring a premium            ┃
┃       subscription.                                                                                             ┃
┃  2 Increase Artist Compensation                                                                                 ┃
┃     • Feedback suggests adjusting the payout structure to better support artists on the platform.               ┃
┃  3 Enhanced Music Playback and Premium Experience                                                               ┃
┃     • Some users expressed a desire for uninterrupted music listening.                                          ┃
┃     • Improvements could include easier upgrades to premium or a new playback mode with fewer interruptions.    ┃
┃  4 Improved User Interface (UI)                                                                                 ┃
┃     • Requests to refine the UI for a more intuitive and aesthetically pleasing user experience.                ┃
┃  5 Expanded Music Library                                                                                       ┃
┃     • Suggestions include adding a broader selection of songs to provide more variety across different genres   ┃
┃       and cultures.                                                                                             ┃
┃  6 Pricing Improvements                                                                                         ┃
┃     • A few users indicated that the current pricing model may be too high and could be made more flexible.     ┃
┃  7 Stability and Bug Fixes                                                                                      ┃
┃     • Reports on specific issues, such as Android system crashes, indicate that addressing these bugs would     ┃
┃       enhance overall app reliability.                                                                          ┃
┃                                                                                                                 ┃
┃ These feature requests offer a range of actionable improvements, from reducing interruptions and making the app ┃
┃ more user-friendly to enhancing overall service value for both users and artists.                               ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [461]:
research_team.print_response("what type of app issues are there?", stream=True)

▰▰▰▰▰▱▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what type of app issues are there?                                                                              ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (48.5s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a detailed hierarchy of app issues, including subcategories and examples covering all relevant themes: ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                              1. Functional Issues                                               ┃
┃                                                                                                                 ┃
┃  • App Crashes and Freezes                                                                                      ┃
┃     • Description: Issues where the app crashes upon startup or during usage, leading to user frustration.      ┃
┃     • Example: “The app keeps freezing and never runs smoothly.”                                                ┃
┃  • Sluggish Performance                                                                                         ┃
┃     • Description: Slow loading times, poor responsiveness, or lag that becomes more prominent after updates.   ┃
┃     • Example: “It's slow an I go on the app it's saying not working.”                                          ┃
┃  • Device and OS Incompatibility                                                                                ┃
┃     • Description: Problems arising on specific phone models or operating systems, causing certain features to  ┃
┃       fail.                                                                                                     ┃
┃     • Example: “The app is broken for Android they fix it and then they break it again.”                        ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                               2. Usability Issues                                               ┃
┃                                                                                                                 ┃
┃  • Complex or Bloated Interface                                                                                 ┃
┃     • Description: Overly complex interfaces with unnecessary features that overwhelm and confuse users.        ┃
┃     • Example: “Incredibly bloated app. So many unnecessary features and it can't do the basics right.”         ┃
┃  • Difficult Navigation                                                                                         ┃
┃     • Description: Poorly designed navigation or layouts that make it hard for users to find features or        ┃
┃       content.                                                                                                  ┃
┃     • Example: “After some updates, it becomes confusing, you need to find your way around again.”              ┃
┃                                                                                                                 ┃
┃ ──────────────────────────────────

In [417]:
research_team.print_response("how to make pasta?", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ how to make pasta?                                                                                              ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (10.2s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Making pasta is a fun and rewarding process. Here's a basic recipe for making fresh pasta from scratch:         ┃
┃                                                                                                                 ┃
┃                                                   Ingredients                                                   ┃
┃                                                                                                                 ┃
┃  • 2 cups all-purpose flour (plus extra for dusting)                                                            ┃
┃  • 3 large eggs                                                                                                 ┃
┃  • 1 tablespoon olive oil                                                                                       ┃
┃  • 1 teaspoon salt                                                                                              ┃
┃                                                                                                                 ┃
┃                                                    Equipment                                                    ┃
┃                                                                                                                 ┃
┃  • Rolling pin or pasta machine                                                                                 ┃
┃  • Knife or pasta cutter                                                                                        ┃
┃  • Clean work surface                                                                                           ┃
┃                                                                                                                 ┃
┃                                                  Instructions                                                   ┃
┃                                                                                                                 ┃
┃  1 Prepare the Dough:                                                                                           ┃
┃     • Place the flour on a clean work surface or in a large bowl. Make a well in the center.                    ┃
┃     • Crack the eggs into the well and add the olive oil and salt.                                              ┃
┃     • Use a fork to gradually mix the flour into the eggs until the dough begins to come together.              ┃
┃  2 Knead the Dough:                                                                                             ┃
┃     • Once the dough has formed, knead it by hand on the floured surface for about 8-10 minutes until smooth    ┃
┃       and elastic.                                                                                              ┃
┃     • Wrap the dough in plastic wrap and let it rest for at least 30 minutes at room temperature. This allows   ┃
┃       the gluten to relax, making it easier to roll out.                                                        ┃
┃  3 Roll Out the Dough:                                                                                          ┃
┃     • Divide the dough into four p

In [391]:
research_team.print_response("what is moon?", stream=True)

▰▰▰▰▰▰▰ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ what is moon?                                                                                                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (57.0s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Below is a detailed summary of the Moon, covering its historical evolution, physical characteristics, cultural  ┃
┃ significance, latest scientific findings, environmental influence, and future exploration plans.                ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                             Historical Perspective                                              ┃
┃                                                                                                                 ┃
┃  • Early Understanding & Cultural Impact:                                                                       ┃
┃    Civilizations from ancient times revered the Moon, integrating its phases into calendars, agricultural       ┃
┃    cycles, and religious ceremonies. Its mysterious glow inspired mythologies and symbolized time, change, and  ┃
┃    femininity across different cultures.                                                                        ┃
┃  • Advancement in Observations:                                                                                 ┃
┃    The introduction of telescopic observations in the 17th century (e.g., by Galileo Galilei) revealed the      ┃
┃    Moon’s mountainous surface and challenged the idea of a perfect celestial body. Later, lunar landing         ┃
┃    missions such as the Apollo programs provided direct samples and a clearer understanding of its geology and  ┃
┃    origins.                                                                                                     ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃                                            Physical Characteristics                                             ┃
┃                                                                                                                 ┃
┃  • Geology and Composition:                                                                                     ┃
┃    The Moon's surface comprises vast basaltic plains (maria) formed by ancient volcanic eruptions and older     ┃
┃    highland regions composed largely of anorthosite. Beneath the surface lies a mantle and a small, partially   ┃
┃    molten core, underscoring its complex geological history.                                                    ┃
┃  • Size and Orbit:                                                                                              ┃
┃    With a diameter of approximately 3,475 kilometers—roughly 27% that of Earth—the Moon orbits our planet at an ┃
┃    average distance of about 384,400 kilometers, completing its orbit in around 27.3 days.                      ┃
┃                                                                                                                 ┃
┃ ──────────────────────────────────

In [386]:
display(Markdown(out))

Below is our comprehensive analysis of customer issues related to playlists based on both quantitative data aggregation and qualitative narrative analysis:

---

## Quantitative Analysis

Our aggregated data from the SQL queries revealed the following common issues:

1. **Shuffle and Song Order Issues**  
   - Track order randomness: Users report that shuffle modes, including “smart shuffle,” often play songs in a non-random or repeated pattern.
  
2. **Playlist Management and Modification**  
   - Unwanted song additions and missing songs: Several customers noted that songs appear or disappear unexpectedly, and in some cases, access restrictions (like premium-only features) affect their experience.

3. **Performance and Functional Errors**  
   - Slow loading and playback errors are frequently reported, indicating that performance and interface glitches disrupt the overall playlist experience.
  
4. **Limitations on Free Version**  
   - Users are dissatisfied with features in the non-premium version (e.g., inability to order songs or download specific tracks).

5. **Advertising and Monetization**  
   - Excess advertising on free accounts is frequently mentioned, negatively impacting the user experience.

---

## Qualitative Analysis

In-depth narrative feedback from customers provided rich detail about their frustrations:

1. **Issues with Playlist Playback**  
   - Users describe playlists playing songs that weren’t originally included. For example:  
     - “Every time I turn on a playlist, it just plays random songs instead of the playlist.”  
     - “When I want to play my Playlist it plays a song that's not on it.”
  
2. **Playlist Accessibility and Deletion Issues**  
   - Some customers have experienced playlists disappearing or becoming inaccessible, even after relying on them for years. Notable comments include:  
     - “I used it for 4 years, saved more than 3000 tracks, and today I log in and nothing plays.”  
     - “My app was working very well, until it decided to let me down and deleted all my playlists.”
  
3. **Impact of Updates**  
   - Recent app updates are blamed for deteriorating playlist functionality, with several users reporting crashes, unplayable playlists, and overall degradation of user experience:  
     - “New update makes some playlists unplayable, and crash the app.”  
     - “Since the last update, I can't access the music from my playlists.”

---

## Summary & Trend Analysis

- **Core Issues Identified:**  
  - Both data sources point to significant challenges in playlist playback integrity and performance, as well as in maintaining user-customized playlists.
  
- **User Experience Concerns:**  
  - The feature limitations in free versions and intrusive advertising are repeatedly criticized. Combined with unexpected behavior changes following software updates, these issues contribute to user dissatisfaction.
  
- **Trends and Recommendations:**  
  - **Shuffle Functionality:** Revisit the shuffle mechanism (especially “smart shuffle”) to ensure truly random playback.
  - **Playlist Integrity:** Enhance the stability of playlist management features to prevent unintended song additions, deletions, or loss of access.
  - **Update Policies:** Implement stricter quality assurance for updates to ensure they do not degrade playlist functionality.
  - **User Segmentation:** Consider separate pathways or feature modes for free versus premium users to alleviate content access issues without compromising core features.

---

## Citations

1. Quantitative analysis results derived from aggregated data records on customer feedback regarding playlist issues.
2. Qualitative feedback extracted from detailed narratives in the dataset. Specific quotations include:  
   - “Every time I turn on a playlist, it just plays random songs instead of the playlist.”  
   - “I used it for 4 years, saved more than 3000 tracks, and today I log in and nothing plays.”

For further reading and evidence-based improvement proposals, please refer to the internal data archives and customer feedback repositories.

---

This synthesis aims to guide further investigation and potential updates to enhance the playlist functionality and overall customer satisfaction.

In [354]:
display(Markdown(refined_query))

Below is a refined version of the query along with a set of research questions organized by historical, current, and future perspectives:

**Refined Research Query:**  
What have been, are currently, and might be the future issues that customers experience with playlists?

**Research Questions to Explore:**

**Historical Perspective**  
- What common issues did early users report regarding playlists?  
- How have customer complaints about playlist curation and organization evolved over time?  
- What design or technical limitations in earlier playlist functionalities contributed to these issues?  
- How did early music streaming platforms respond to and address customer feedback about playlists?

**Current Perspective**  
- What are the most frequently reported customer issues with playlists in today’s streaming services?  
- How effective are current personalization and recommendation algorithms in addressing user needs?  
- What usability challenges (e.g., editing, sharing, or organizing playlists) do customers currently face?  
- How do different demographics perceive and interact with playlists today?

**Future Perspective**  
- What potential issues could emerge as music streaming technology and user expectations evolve?  
- How might advances in artificial intelligence and machine learning impact playlist curation, and what new challenges might that introduce?  
- What future design considerations should be prioritized to prevent emerging customer issues with playlists?  
- How could changing consumption patterns (e.g., the rise of short-form content or increased interactivity) shape future customer expectations and related challenges?

This structured set of questions can help guide a comprehensive research study on customer issues with playlists from a historical, current, and future perspective.

In [ ]:

# # Synthesis Team: Compiles the final report.
# synthesis_team = Team(
#     mode="coordinate",
#     members=[
#         synthesis_agent,
#         # fact_checker_agent,
#         # critique_agent,
#     ],
#     model=OpenAIChat(id="o3-mini"),
#     success_criteria="A final deep research report with clear, well-organized insights and verified facts.",
#     instructions=["Synthesize the research inputs into a coherent report"],
#     show_tool_calls=False,
#     markdown=True,
# )

# ---------------------------
# Main Reasoning Agent with Hybrid Approach
# ---------------------------
class MainReasoningAgentSelective:
    """
    Uses the decision agent to selectively invoke sub-agents,
    coordinates with a research team for data gathering, and then
    uses a synthesis team for final report generation.
    """
    def __init__(self, decision_agent, research_team, synthesis_team, query_refiner):
        self.decision_agent = decision_agent
        self.research_team = research_team
        self.synthesis_team = synthesis_team
        self.query_refiner = query_refiner
        # self.evaluator = TournamentEvaluatorAgent()

    def run(self, query):
        # Step 0: Decide which agents are needed.
        # decisions = self.decision_agent.decide(query)
        # print("[DecisionAgent] Selected agents:", decisions)
        
        # Step 1: Refine query if needed.
        # if "queryrefineragent" in decisions:
        refined = self.query_refiner.run(query).content.strip()
        # else:
        #     refined = query
        print("[Main] Refined Query:", refined)
        
        # Step 2: Run the research team if either internal or external research is needed.
        # research_needed = any(agent in decisions for agent in ["internalresearchagent", "externalresearchagent"])
        # if research_needed:
        #     research_input = f"Research query: {query}\nRefined query: {refined}"
        #     research_response = self.research_team.run(research_input)
        #     research_summary = research_response.content.strip()
        # else:
        #     research_summary = ""
        research_input = f"Research query: {query}\nRefined query: {refined}"
        research_response = self.research_team.run(research_input)
        research_summary = research_response.content.strip()
        # # Step 3: Synthesize the final report using the synthesis team.
        # synthesis_input = (
        #     f"Original Query: {query}\n"
        #     f"Refined Query: {refined}\n\n"
        #     f"Research Summary: {research_summary}\n"
        # )
        # print(synthesis_input)
        # synthesis_response = self.synthesis_team.run(synthesis_input)
        # report = synthesis_response.content.strip()
        report = research_summary
        print("[Main] Final Synthesis Completed.")
        
        return report


main_agent = MainReasoningAgentSelective(
    decision_agent=decision_agent,
    research_team=research_team,
    synthesis_team=synthesis_team,
    query_refiner=query_refiner_agent,
)


In [244]:
# Team?

In [255]:
research_query = "what is ghibili art?"
final_report = main_agent.run(research_query)


display(Markdown(final_report))

[Main] Refined Query: Below is a refined research plan for the query "What is Ghibli art?" analyzed from historical, current, and future perspectives. Each section includes key questions that should be answered:

---

### Historical Perspective
- What are the origins of Ghibli art, and how did it develop over time?
- Which influences (historical art movements, traditional techniques, cultural elements) shaped the early visual style of Studio Ghibli?
- Who were the key artists and directors that contributed to defining the Ghibli art style?
- What distinguishes early Ghibli works from other animation styles in history?

---

### Current Perspective
- What are the core characteristics and techniques of Ghibli art as seen in contemporary works?
- How is Ghibli art defined in the context of modern animation and visual storytelling?
- In what ways are current artists influenced by or adapting the Ghibli art style in their own works?
- How is the legacy of Ghibli art preserved, celebrated, o

Below is the comprehensive research report on "What is Ghibli art?" organized by historical, current, and future perspectives with key questions addressed, supporting evidence, and relevant citations.

---

## Historical Perspective

### Origins and Development
- **Founding and Early Works:**  
  Studio Ghibli was founded in 1985 by Hayao Miyazaki, Isao Takahata, and Toshio Suzuki. Early films like *My Neighbor Totoro* (1988) and *Kiki's Delivery Service* (1989) established the studio’s reputation for hand-drawn animation that puts great emphasis on storytelling and detailed artistry.

### Influences from Historical Art Movements and Cultural Elements
- **Traditional Japanese Art:**  
  Ghibli art draws significant influence from traditional Japanese aesthetics. The atmospheric landscapes and delicate line work found in the woodblock prints of Hasui Kawase (a key figure in the shin-hanga movement) mirror the studio’s portrayal of nature.  
  Citation: [Far Out Magazine](https://faroutmagazine.co.uk/hasui-kawase-artist-inspired-studio-ghibli/?utm_source=openai)
  
- **Western Art Movements:**  
  Exposure to Western works, such as John Everett Millais's *Ophelia* (1851–52) from the Pre-Raphaelite Brotherhood, influenced Miyazaki’s approach. The attention to detail and the play of light in such paintings contributed to the realism and depth that characterize Ghibli’s early animation style.  
  Citation: [Hasta St. Andrews](https://www.hasta-standrews.com/features/2022/10/26/ophelia-and-ponyo-millais-and-miyazaki-how-the-pre-raphaelite-brotherhood-influenced-the-animated-films-of-studio-ghibli?utm_source=openai)

### Key Artists and Directors
- **Hayao Miyazaki and Isao Takahata:**  
  Miyazaki is widely recognized as the principal visual architect behind Ghibli’s style with his distinctive sketches and watercolor techniques. Takahata contributed with his unique narrative approaches, ensuring a complementary balance to the visual storytelling.

### Distinctions from Other Animation Styles
- **Hand-drawn Commitment:**  
  While many studios began transitioning to digital animation during the late 20th century, Studio Ghibli maintained a commitment to hand-drawn techniques. This choice resulted in a warm, organic aesthetic that set it apart from the emerging mechanical feel of digital animations.

---

## Current Perspective

### Core Characteristics and Techniques
- **Blend of Traditional and Digital:**  
  Modern Ghibli productions still emphasize hand-drawn animation, characterized by painterly backgrounds, fluid movement, and a harmonious color palette. However, digital tools are occasionally integrated (as seen in *The Boy and the Heron* (2023)) to enhance visual effects while retaining the hand-drawn quality.
  Citation: [Digital Arts Blog](https://www.digitalartsblog.com/artist-spotlights/hayao-miyazaki?utm_source=openai)

### Ghibli Art in Contemporary Animation and Visual Storytelling
- **Emotional Depth and Narrative Focus:**  
  In today’s animation landscape, Ghibli art is synonymous with deeply emotional storytelling and vivid visual narratives. Its influence is broadly observed, inspiring artists around the globe.

### Influence on Modern Artists
- **New-Generation Inspiration:**  
  Contemporary artists like Julien Ceccaldi (with his comic *Solito* (2018)) and Pippa Dyrlaga (known for her paper-cut works) incorporate stylistic cues reminiscent of Ghibli films.  
  Citation: [Artsy](https://www.artsy.net/article/artsy-editorial-studio-ghibli-inspired-new-generation-artists?utm_source=openai)

### Preservation and Adaptation in the Global Creative Scene
- **Longevity and Adaptation:**  
  Ghibli’s legacy is celebrated internationally through exhibitions, academic studies, and fan art. At the same time, the rise of AI-generated art inspired by Ghibli’s style has sparked debates over authenticity and copyright, with Miyazaki himself voicing concerns.  
  Citation: [AP News](https://apnews.com/article/0f4cb487ec3042dd5b43ad47879b91f4?utm_source=openai)

---

## Future Perspective

### Evolving with Digital Technology
- **Balancing Tradition and Innovation:**  
  As technological advancements continue to reshape animation, Studio Ghibli faces a dual challenge: innovating with digital techniques while preserving their cherished hand-drawn aesthetic. Movies like *Earwig and the Witch* (2020) showcase early tests with CGI, though they received mixed responses.

### Influence on Future Animation Trends
- **Ongoing Inspiration:**  
  The emotional depth and artistic style of Ghibli films are expected to continue influencing global animation. Future creators may merge traditional hand-drawn methods with emerging digital tools to create compelling, emotionally resonant works.

### Redefinition through Global Influences
- **Cross-Cultural Collaborations:**  
  With Ghibli’s universal appeal, future adaptations might see collaborations across cultures, blending diverse artistic traditions with the established Ghibli visual language. This fusion could lead to innovative narratives and aesthetics that reflect both global trends and the rich heritage of Studio Ghibli.

### Studio Ghibli’s Continuing Role
- **Steering Future Narratives:**  
  As the studio evolves, it will likely continue to shape animated storytelling by balancing innovation with its traditional practices. This ongoing influence will help secure Ghibli's legacy and inspire future generations to explore new visual and narrative frontiers.
  
---

## Additional Insights on Ethical Debates
The rise of AI tools capable of replicating the Ghibli art style has led to ethical debates within the creative community. Concerns include issues of copyright and the authenticity of art generated by AI, with notable media coverage addressing these matters:
- [Hayao Miyazaki's AI Nightmare – The Atlantic](https://www.theatlantic.com/newsletters/archive/2025/03/studio-ghibli-memes-openai-chatgpt/682235/?utm_source=openai)
- [ChatGPT's Viral Studio Ghibli-style Images Highlight AI Copyright Concerns – AP News](https://apnews.com/article/0f4cb487ec3042dd5b43ad47879b91f4?utm_source=openai)
- [New ChatGPT Update Spurs Flood of Ghibli-style Portraits – Axios](https://www.axios.com/2025/03/26/chatgpt-images-ghibli-portraits?utm_source=openai)

---

## Summary

Ghibli art is a unique visual language rooted in a rich blend of traditional Japanese art and Western influences, defined by its hand-drawn animation and attention to detail. Historically, it was shaped by the efforts of key artists like Hayao Miyazaki and cultural influences such as the shin-hanga movement and Pre-Raphaelite art. In the contemporary scene, while Studio Ghibli continues to honor its hand-drawn legacy, it is gradually incorporating digital methods without compromising the inherent warmth of its art style. Looking forward, Ghibli art is poised to influence future trends by merging tradition with innovation—an influence that will likely be amplified through global collaborations and ongoing debates about the role of AI in art.

This comprehensive view underscores the enduring appeal and dynamic evolution of Ghibli art, ensuring its continued relevance in the world of animation.

---

In [242]:
research_query = "what is graphrag?"
final_report = main_agent.run(research_query)


display(Markdown(final_report))

[Main] Refined Query: Below is a refined set of research questions for investigating "graphrag" from a past, present, and future perspective:

- **Clarification & Definition:**
  - What is graphrag and how is it defined?
  - Which disciplines or fields does graphrag relate to?

- **Historical Context (Past):**
  - When and why was graphrag first introduced or developed?
  - What problems or needs was graphrag initially designed to address?
  - How has the definition or scope of graphrag evolved since its inception?

- **Contemporary Use & Impact (Present):**
  - How is graphrag currently applied or utilized in its relevant fields?
  - What are the key features, functions, or methodologies associated with graphrag today?
  - What are the current limitations or challenges faced by graphrag in practice?

- **Future Directions & Developments:**
  - What emerging trends or technologies could influence the evolution of graphrag?
  - How might graphrag be improved or expanded to meet future r

ERROR    API status error from OpenAI API: Error code: 400 - {'error': {'message': "Response format 'json_object'  
         is not supported with web_search.", 'type': 'invalid_request_error', 'param': 'response_format', 'code':  
         None}}

WARNING  Attempt 1/1 failed: Response format 'json_object' is not supported with web_search.

ERROR    Failed after 1 attempts. Last error using OpenAIChat(gpt-4o-search-preview)

ERROR    Reasoning error: Response format 'json_object' is not supported with web_search.

[Main] Final Synthesis Completed.


Below is a comprehensive summary addressing the refined research questions on GraphRAG from a historical, contemporary, and future perspective.

---

## 1. Clarification & Definition

- **What is GraphRAG?**  
  GraphRAG is an advanced Retrieval-Augmented Generation (RAG) framework that integrates knowledge graphs into the traditional RAG approach. Instead of solely relying on unstructured text and vector similarity, GraphRAG leverages structured representations of entities and their interrelationships. This method leads to an enriched contextual understanding and improved reasoning capabilities in language models.

- **Which Disciplines or Fields Does It Relate To?**  
  GraphRAG is interdisciplinary and intersects with several fields:
  - **Artificial Intelligence (AI):** Enhancing the reasoning and contextual understanding of AI models.
  - **Natural Language Processing (NLP):** Improving response generation and question answering through structured data.
  - **Data Science:** Leveraging structured data representations for more precise information retrieval.
  - **Knowledge Management:** Organizing complex information networks using knowledge graphs.

> **Citations:**  
> [Microsoft GraphRAG Documentation](https://microsoft.github.io/graphrag/?utm_source=openai)

---

## 2. Historical Context (Past)

- **Introduction and Development:**  
  GraphRAG emerged as an enhancement over traditional RAG systems to overcome limitations in handling complex queries. The integration of structured knowledge graphs was designed to connect disparate pieces of information more effectively and synthesize insights that simple retrieval systems could not.

- **Initial Problems Addressed:**  
  Traditional RAG methods faced challenges such as:
  - Isolating semantically related pieces of data that were dispersed throughout large documents.
  - Addressing multi-hop reasoning issues where connecting context across data sources was difficult.
  
- **Evolution in Definition and Scope:**  
  Over time, GraphRAG evolved to include:
  - **Hierarchical Clustering:** Methods such as the Leiden technique to build community hierarchies.
  - **Community Summarization:** Creating summaries for groups (communities) of related entities.
  
> **Citations:**  
> [Microsoft GraphRAG Documentation](https://microsoft.github.io/graphrag/?utm_source=openai)

---

## 3. Contemporary Use & Impact (Present)

- **Current Applications:**  
  GraphRAG is currently employed in various settings:
  - **Question Answering:** Providing contextually accurate responses by drawing on structured entity relationships.
  - **Summarization:** Extracting key themes and creating coherent summaries of lengthy texts.
  - **Dialogue Systems:** Enhancing virtual assistants and AI chatbots by delivering context-aware responses.
  - **Knowledge Extraction:** Useful in sectors like healthcare, legal research, and academia for synthesizing complex information.

- **Key Features and Methodologies:**  
  Today's implementations of GraphRAG typically include:
  - **Knowledge Graph Construction:** Building structured graphs from raw textual data.
  - **Hierarchical Clustering:** Organizing entities into communities to enhance retrieval efficiency.
  - **Community Summarization:** Generating digestible summaries for clusters within the graph.
  - **Enhanced Information Retrieval:** Leveraging structural relationships to improve the relevance and accuracy of retrieval outputs.
    
- **Current Limitations and Challenges:**  
  Despite its advantages, several challenges remain:
  - **Graph Construction Complexity:** Creating meaningful and accurate graphs can be resource-intensive.
  - **Computational Demands:** Processing large, complex graphs requires significant computational power, especially for real-time applications.
  - **Data Dependency:** The performance of GraphRAG heavily depends on the quality and completeness of the underlying dataset.

> **Citations:**  
> [GeeksforGeeks on GraphRAG](https://www.geeksforgeeks.org/what-is-graphrag/?utm_source=openai)

---

## 4. Future Directions & Developments

- **Emerging Trends and Technologies:**  
  Future enhancements in GraphRAG are likely to include:
  - **Automated Knowledge Graph Construction:** Using sophisticated machine learning algorithms to minimize manual efforts.
  - **Multimodal Integration:** Incorporating data from various formats (e.g., text, images, sensor data) to provide richer context.
  - **Real-Time Graph Updates:** Developing dynamic system capabilities to handle rapidly changing datasets, which is crucial in fields like finance and social media.

- **Potential Improvements and Expansions:**  
  Prospective developments may focus on:
  - **Scalability:** Modular graph designs and cloud-based processing to manage large-scale applications.
  - **Explainable AI:** Building frameworks that offer transparency and trust, particularly in sensitive environments like healthcare.
  - **Domain-Specific Ontologies:** Creating tailored ontologies to enhance the applicability and accuracy within specific industries.

- **Potential Applications and Innovations:**  
  Looking ahead, GraphRAG could play an important role in:
  - **Healthcare:** Linking patient histories with medical research for personalized treatment options and better diagnostic accuracy.
  - **Finance:** Uncovering fraud patterns and improving risk management through systematic transaction analysis.
  - **E-commerce:** Refining recommendation systems by linking customer behavior with product data.
  - **Legal Research:** Streamlining case law analysis by connecting statutes and legal precedents.

> **Citations:**  
> [Chitika on GraphRAG Developments](https://www.chitika.com/graphrag-origin-uses-implementation/?utm_source=openai)

---

## Summary

GraphRAG represents a significant leap forward in the field of Retrieval-Augmented Generation by incorporating structured knowledge graphs into AI systems. This integration enhances the system's ability to reason over complex datasets, synthesize information across disparate data sources, and generate more contextually accurate responses. Initially developed to overcome limitations of traditional RAG approaches, GraphRAG has evolved through the incorporation of hierarchical clustering and community summarization. Today, its applications span from question answering and summarization to advanced dialogue systems and knowledge extraction in various industries. Looking forward, innovations such as automated graph construction, multimodal data integration, and real-time updates promise to expand its capabilities further, paving the way for its adoption in critical areas like healthcare, finance, e-commerce, and legal research.

---

The above summary integrates historical context, present applications, and future trends, supported by relevant citations and links for further reading.

In [215]:
from IPython.display import Markdown, display

display(Markdown(final_report))

Below is a structured summary addressing the research query “What is graphrag?” from historical, current, and future perspectives.

---

## Historical Perspective

**Definition & Origins:**  
GraphRAG (Graph-based Retrieval-Augmented Generation) builds on the concept of Retrieval-Augmented Generation (RAG) introduced to help mitigate issues like hallucinations and outdated information in large language models (LLMs). The early RAG approaches relied on external knowledge bases to refine responses. However, they could not efficiently handle complex relational data. To overcome this, GraphRAG was developed by integrating graph-based knowledge representations—where entities are viewed as nodes and their relationships as edges—to better capture and utilize structured interconnections in data.  

**Evolution Over Time:**  
- **Early Foundations:** The original RAG framework provided insights into combining retrieval systems with generative models, but with limited ability to manage intricate relational information.  
- **Pioneering Contributions:** Innovations in graph-based structures were introduced by researchers and institutions (such as Microsoft Research) that extended the RAG model to incorporate graph traversal techniques, thereby significantly enhancing the precision and context awareness of AI outputs.  

*Citation:*  
- [arXiv Preprint on GraphRAG](https://www.arxiv.org/abs/2408.08921?utm_source=openai)

---

## Current Perspective

**Contemporary Definition & Applications:**  
GraphRAG is now defined as a framework that fuses graph-based indexing of data with retrieval-augmented generation. It is used to integrate structured knowledge from complex datasets with the dynamic and adaptive generation capabilities of modern LLMs. This provides an edge in applications such as:  
- **Customer Support:** Enhancing automated troubleshooting by navigating through structured product knowledge graphs.  
- **Sales and Marketing:** Automating personalized recommendations by mapping customer journeys through graph structures.  
- **Healthcare:** Informing diagnostics by traversing patient history and clinical research graphs to personalize treatment strategies.

**Methodologies & Technological Approaches:**  
- **Graph-Based Knowledge Representation:** Constructs a knowledge graph where entities and their relationships are explicitly modeled, providing a clear and interconnected structure for data retrieval.  
- **Graph-Guided Retrieval:** Enhances the retrieval step by using graph structure to provide contextually rich and precise pieces of information that feed into a generative model.  
- **Graph-Enhanced Generation:** Leverages LLMs to generate responses that are better informed by the underlying graph structure, leading to responses that are both accurate and context-sensitive.

**Leading Institutions:**  
Prominent research institutions such as Microsoft Research are actively developing and deploying GraphRAG systems, with implementations and proofs-of-concept available on platforms like GitHub.

*Citation:*  
- [Arion Research Blog on GraphRAG](https://www.arionresearch.com/blog/graphrag-the-future-of-knowledge-driven-ai?utm_source=openai)  
- [Microsoft Research Blog on GraphRAG](https://www.microsoft.com/en-us/research/blog/graphrag-new-tool-for-complex-data-discovery-now-on-github/?utm_source=openai)

---

## Future Directions

**Potential Innovations & Gaps:**  
Despite its advances, GraphRAG still encounters challenges that inspire active research and innovation:  
- **Dynamic and Adaptive Graphs:** Development of systems for real-time updates so that knowledge graphs can accommodate new data and evolving relationships.  
- **Multi-Modal Data Integration:** Expanding GraphRAG to include diverse data types (e.g., images, audio, video) to provide a richer, more holistic context.  
- **Scalable Retrieval Mechanisms:** Enhancing algorithms to manage increasingly larger graphs with millions or billions of nodes efficiently.  
- **Combining with Graph Foundation Models:** Integrating advanced models specifically built for graph data to further improve reasoning and context extraction.
- **Lossless Compression Techniques:** Refinement of methods to compress extensive context data without sacrificing critical information during processing.
- **Standardized Benchmarks:** Establishing unified benchmarks to validate and compare GraphRAG implementations objectively.

**Broader Implications:**  
The continual advancement of GraphRAG is expected to extend its utility into diverse fields such as finance, legal compliance, and smart city applications. As the approach matures, it promises to enhance both theoretical models and practical technologies by offering solutions that are increasingly precise, explainable, and context-aware.

*Citation:*  
- [arXiv Preprint on GraphRAG Future Directions](https://arxiv.org/html/2408.08921?utm_source=openai)

---

## Summary

GraphRAG represents a significant evolution in the integration of graph-based representations with retrieval-augmented generation techniques. Historically, it emerged to address the shortcomings of conventional RAG models by embracing structured knowledge, allowing for a more contextual understanding of relationships. Today, its applications span various domains—from customer service enhancement to healthcare analytics—driven by methodologies that fuse graph indexing with modern LLMs. Future research is poised to refine these capabilities further, particularly around real-time updates, multi-modality integration, scalability, and the establishment of standardized evaluation benchmarks.

This comprehensive overview provides a solid foundation for understanding the past evolution, current state, and future potential of GraphRAG.

In [224]:
research_query = "what are different type of llm agents?"
final_report = main_agent.run(research_query)
display(Markdown(final_report))

[Main] Refined Query: Below is a refined research query with a plan that takes into account past, present, and future perspectives, along with a list of subquestions that can guide your investigation.

---

## Refined Research Query

What are the different types of LLM agents, how have they evolved over time, what are their current characteristics, and what future developments might we expect?

---

## Key Research Questions

### Historical Perspective (Past)
- **Definition and Origin:**
  - What exactly do we mean by "LLM agents"?
  - When and how did the concept of LLM agents first emerge?
- **Evolution of Agent Architectures:**
  - How have LLM agents evolved in terms of design and functionality over time?
  - What were the foundational models or architectures that paved the way for modern LLM agents?

### Current State (Present)
- **Categories and Characteristics:**
  - What are the main types of LLM agents currently in use (e.g., autonomous agents, chain-of-thought agents, retriev

Below is the final comprehensive research summary on the different types of LLM agents, their evolution over time, current characteristics, and potential future developments.

---

# Comprehensive Report on LLM Agents: Past, Present, and Future

## 1. Introduction

Large Language Model (LLM) agents have undergone dramatic evolution—from early rule-based systems to today’s highly sophisticated models. This report provides an in‐depth analysis of LLM agents covering historical evolution, current categories and applications, the technical and ethical challenges they face, and the future directions in research and implementation.

---

## 2. Historical Perspective

### Definition and Origin
- **Definition:** LLM agents are intelligent systems that interact with their environment—in many cases via natural language—by processing, generating, and reasoning with text.  
- **Origins:** Early agents such as ELIZA (1966) laid the groundwork with simple conversational capabilities; these systems originally operated through rule-based interactions.

### Evolution of Agent Architectures
- **Text Agents:** Early systems used fixed, rule-based approaches resulting in limited conversational scope.
- **Transition to LLMs:** The advent of large-scale pretrained models (e.g., GPT, BERT) marked a significant evolution towards context-aware and versatile agents.
- **Reasoning Agents:** More recent developments integrate internal reasoning mechanisms and decision-making processes that allow agents to function autonomously across domains.

*Reference: [Brief History of LLM Agents – Medium](https://medium.com/%40razgaleh/brief-history-of-llm-agents-d7d22f82a539?utm_source=openai)*

---

## 3. Current State

### Categories and Characteristics

**1. Conversational Agents:**  
   - Designed for general dialogue and interaction (e.g., ChatGPT).

**2. Task-Oriented Agents:**  
   - Specialized systems fine-tuned to perform specific roles such as customer service, legal assistance, or medical advisories.

**3. Creative Agents:**  
   - Capable of generating text, art, music, and even programming code, enhancing creative processes.

**4. Collaborative Agents:**  
   - Work alongside human teams to support decision-making and coordinate tasks.

**5. Multimodal Agents:**  
   - Process and generate various types of media (text, audio, images), broadening their application range.

**6. Autonomous Agents:**  
   - Operate with minimal human intervention by making independent decisions.

**7. Multi-Agent Systems:**  
   - Involve several agents working together to solve complex problems.

### Applications
- **Retail:** Automated customer support and enhanced user experiences.
- **Education:** Personalized learning and individualized tutoring systems.
- **Enterprise:** Streamlining operations through workflow automation and data analysis.
- **Creative Industries:** Content generation in media, arts, and programming.

### Evaluation Metrics
- **Accuracy, Coherence, Relevance, and Efficiency:**  
  Performance is measured through these key metrics, ensuring that responses are correct, logically consistent, contextually relevant, and resource-effective.

### Technical and Ethical Considerations
- **Technical Challenges:**  
  Scalability, computational resource demands, biases from training data, and interpretability of decision processes.
- **Ethical Challenges:**  
  Privacy issues, potential spread of misinformation, and the need to ensure fairness and transparency in automated decision-making.

*References: [Types of LLM Agent – Blog](https://blog.promptlayer.com/types-of-llm-agent/?utm_source=openai), [Understanding LLM Agents – Ciklum](https://www.ciklum.com/resources/blog/understanding-llm-agents?utm_source=openai), [History and Evolution – GeeksforGeeks](https://www.geeksforgeeks.org/history-and-evolution-of-llms/?utm_source=openai)*

---

## 4. Future Directions

### Innovation and Advancements
- **Enhanced Model Architectures:**  
  Designing more efficient architectures using techniques like model distillation, sparse representations, and modular designs.
- **Hybrid and Multimodal Integration:**  
  Future LLM agents will likely integrate text, images, and audio inputs along with real-time data feeds for more robust responses.
- **Personalization:**  
  Adaptation through continuous and context-based learning, leading to highly personalized agent responses over time.

### Regulatory Considerations
- **Comprehensive Regulatory Frameworks:**  
  Developments in regulation are expected to emphasize transparency, data privacy, and ethical guidelines—ensuring responsible AI deployment.
- **Global Coordination:**  
  Cross-jurisdictional initiatives will aim to harmonize standards globally to support innovation while maintaining accountability.
- **Mandatory Transparency and Audits:**  
  Future guidelines may require detailed documentation of data sources, training methods, and decision-making processes in AI systems.

*Reference: [OECD AI Regulatory Initiatives](https://www.oecd.org/going-digital/ai-principles/) and [World Economic Forum on AI Governance](https://www.weforum.org/agenda/2020/09/artificial-intelligence-ethical-governance/)*

### Societal Impact
- **Employment Transformation:**  
  While automation may change traditional job roles, new positions in AI oversight, ethics, and management are expected to emerge.
- **Educational Access:**  
  LLM agents have the potential to democratize education by providing widespread access to personalized, high-quality instructional content.
- **Public Trust and Bias Mitigation:**  
  Ongoing research and social dialogue are vital for addressing bias and ensuring ethical practices, thereby strengthening public trust in AI.

*Reference: [Societal Implications – World Economic Forum](https://www.weforum.org/agenda/2020/09/artificial-intelligence-ethical-governance/)*

---

## 5. Summary and Concluding Remarks

LLM agents have significantly evolved—from early rule-based systems to today’s multi-faceted, context-aware and autonomous agents. Key points include:

- **Historical Evolution:**  
  Development has spanned rule-based systems, neural network-based models, to advanced LLM agents capable of reasoning.
- **Current State:**  
  Diverse agent categories now support various applications ranging from customer service to creative and data analysis roles.
- **Challenges:**  
  Despite significant progress, technical challenges (e.g., scalability, bias) and ethical issues (e.g., privacy, transparency) remain central.
- **Future Directions:**  
  Innovation in architecture, multimodal integration, and personalization will drive future capabilities, while rigorous regulatory frameworks and social considerations will be essential to ensure ethical deployment and public trust.

Balancing technological innovation with ethical and regulatory oversight will be crucial to harness the full potential of LLM agents as they continue to reshape various societal and industrial domains.

---

## 6. References

- [Transformers: "Attention Is All You Need"](https://ai.googleblog.com/2017/08/transformers-are-all-you-need.html)
- [Brief History of LLM Agents – Medium](https://medium.com/%40razgaleh/brief-history-of-llm-agents-d7d22f82a539?utm_source=openai)
- [Types of LLM Agent – Blog](https://blog.promptlayer.com/types-of-llm-agent/?utm_source=openai)
- [Understanding LLM Agents – Ciklum](https://www.ciklum.com/resources/blog/understanding-llm-agents?utm_source=openai)
- [History and Evolution – GeeksforGeeks](https://www.geeksforgeeks.org/history-and-evolution-of-llms/?utm_source=openai)
- [Ethical AI Considerations – IBM Watson](https://www.ibm.com/watson/assets/duo/pdf/ethical-ai.pdf)
- [Global AI Regulatory Initiatives – OECD](https://www.oecd.org/going-digital/ai-principles/)
- [World Economic Forum on AI Governance](https://www.weforum.org/agenda/2020/09/artificial-intelligence-ethical-governance/)

---

This research offers a thorough perspective on LLM agents, illuminating their past achievements, current applications, and future prospects. The combined evolution in technology and thoughtful regulatory planning will be key to leveraging these advancements while ensuring ethical and societal well-being.

In [218]:
research_query = "what are different type of llm agents?"
final_report = main_agent.run(research_query)
display(Markdown(final_report))

[Main] Refined Query: Below is a refined version of the query along with a research plan that outlines key questions addressing past developments, current research, and future directions:

---

### Refined Research Query
What are the different types of LLM agents, and how have they evolved over time in terms of design, functionality, and application areas?

---

### Research Questions to Explore

1. **Definition and Scope**
   - What exactly are LLM agents, and how are they defined in the literature?
   - What are the core functionalities that distinguish LLM agents from other AI systems?

2. **Taxonomy and Classification**
   - What are the main categories or types of LLM agents currently identified?
   - How do these categories differ in terms of design structure (e.g., reactive, deliberative, planning-based, chain-of-thought, etc.)?
   - What criteria are used to classify LLM agents?

3. **Historical Perspective**
   - How have LLM agents evolved from early rule-based systems to cur

Below is a comprehensive summary of the current research on different types of LLM agents, incorporating historical evolution, taxonomy, methodological approaches, and future directions along with citations.

---

## Overview of LLM Agents

**LLM (Large Language Model) agents** are AI systems that integrate the language processing power of large-scale models with autonomous decision-making. They go beyond simple text generation by interacting with environments, making decisions, and even learning from real-time interactions. This dynamic functionality enables them to be deployed across a range of applications—from conversational assistants to specialized decision-support systems.

---

## 1. Definition and Scope

- **Definition:**  
  LLM agents are defined as AI systems that use state-of-the-art language models to interpret natural language inputs, generate outputs, and autonomously take actions based on those inputs. Their core functionalities include understanding complex instructions, engaging in step-by-step reasoning, and often incorporating methods of planning and decision-making.

- **Core Functionalities:**  
  - Processing natural language inputs to understand context.  
  - Autonomous decision-making and planning.  
  - Integration of external knowledge bases or real-time data.  
  - Adaptability to various tasks, from simple queries to complex problem-solving.

---

## 2. Taxonomy and Classification

LLM agents can be classified by examining their design architecture and operational paradigms:

- **Reactive Agents:**  
  - Quickly respond to inputs with pre-programmed or learned responses without maintaining a rich internal state.

- **Deliberative Agents:**  
  - Engage in planning and reasoning, considering future consequences before acting.

- **Planning-Based Agents:**  
  - Use internal models of the environment to plan action sequences to achieve specific objectives.

- **Chain-of-Thought Agents:**  
  - Implement a step-by-step reasoning process to break down complex queries, which enhances accuracy and interpretability.

*Sources:*  
- [Springer Link on LLM agents](https://link.springer.com/article/10.1007/s10462-024-10888-y?utm_source=openai)  
- [Datadna blog post](https://www.datadna.in/post/navigating-the-new-types-of-llm-agents-and-architectures?utm_source=openai)

---

## 3. Historical Evolution

The evolution of LLM agents can be traced in several stages:

1. **Rule-Based Systems:**  
   - Early AI systems that strictly followed handcrafted rules.
   - Limitation: Rigid, lacking adaptability and scalability.

2. **Reinforcement Learning (RL) Agents:**  
   - Introduced the concept of learning from interactions.
   - Offered adaptability but required substantial training and struggle with generalization.

3. **Integration with Large Language Models:**  
   - LLMs provided advanced natural language understanding.
   - Enabled the emergence of agents capable of autonomous reasoning, planning, and dynamic problem-solving.

*Sources:*  
- [Medium article on historical evolution](https://saurabhharak.medium.com/llm-agents-their-past-present-and-future-22988c29a5f8?utm_source=openai)

---

## 4. Current Landscape

LLM agents today are used in several key application domains:

- **Generative AI:**  
  - Tools like OpenAI's Codex assist in code generation.
  
- **Conversational AI:**  
  - Systems such as Google’s LaMDA power advanced chatbots and virtual assistants, delivering context-aware and natural dialogues.

- **Business Automation:**  
  - Employed in business applications like customer support, recruitment, and administrative processes, driving new SaaS products and transforming traditional workflows.

*Sources:*  
- [Financial Times article on AI agents](https://www.ft.com/content/36785ec8-6f9f-455f-ac74-645bcaa9e221?utm_source=openai)

---

## 5. Methodological Considerations

Key aspects involved in the construction of effective LLM agents include:

- **Data Quality:**  
  - High-quality, diverse, and unbiased training data is essential for robust performance and generalization.

- **Model Architectures:**  
  - Choosing suitable architectures (e.g., transformer-based or hybrid models) to balance performance with computational efficiency.

- **Ethical Alignment:**  
  - Incorporating ethical safeguards and bias mitigation strategies to ensure trustworthiness and fairness in decision-making.

*Sources:*  
- [Springer Link detailed perspective](https://link.springer.com/article/10.1007/s10462-024-10888-y?utm_source=openai)

---

## 6. Key Challenges

Several crucial challenges remain for LLM agents:

- **Interpretability:**  
  - Complex decision-making processes can limit the clarity of how responses are generated, affecting trust and accountability.

- **Reliability:**  
  - Continuous and accurate performance in varied and sometimes unpredictable scenarios remains a challenge.

- **Bias:**  
  - Addressing inherent biases in training data is critical to avoid reinforcing harmful stereotypes.

- **Scalability:**  
  - The high computational costs of developing and deploying large models pose practical limitations.

*Sources:*  
- [Springer Link on evaluation frameworks](https://link.springer.com/article/10.1007/s10462-024-10888-y?utm_source=openai)

---

## 7. Future Directions and Research Opportunities

Future research in LLM agents is likely to focus on several promising areas:

- **Personalization:**  
  - Developing agents that adapt dynamically to individual user needs and preferences.

- **Multimodal Integration:**  
  - Combining language processing with vision, audio, and other sensory inputs to create more versatile agents.

- **Federated Learning:**  
  - Employing decentralized learning methods to enhance data privacy and collaborative model improvement.

- **Fusion with Other AI Fields:**  
  - Integrating advances from reinforcement learning, meta-learning, and explainability research to build more robust and adaptable agent designs.

*Sources:*  
- [Datadna post on future trends](https://www.datadna.in/post/navigating-the-new-types-of-llm-agents-and-architectures?utm_source=openai)

---

## Additional References

- [FT – The future of AI agents](https://www.ft.com/content/36785ec8-6f9f-455f-ac74-645bcaa9e221?utm_source=openai)  
- [Reuters – Autonomous agents forecast](https://www.reuters.com/technology/artificial-intelligence/autonomous-agents-profitability-dominate-ai-agenda-2025-executives-forecast-2024-12-12/?utm_source=openai)

---

## Conclusion

LLM agents are a transformative evolution in AI, combining the sophisticated language understanding of LLMs with robust decision-making capabilities. As research and development continue, overcoming challenges such as interpretability, reliability, and bias will be critical. Advances in personalization, multimodal integration, and federated learning are set to further expand the capabilities and applications of these agents.

This comprehensive overview at the intersection of historical context, current applications, and future research directions provides a solid framework for understanding and advancing LLM agents.

---

In [178]:
print(final_report)

Here is a detailed research report on RAG systems, focusing on GraphRAG, with verified facts:

---

# Research Report on GraphRAG: Retrieval-Augmented Generation Systems

## Overview

Retrieval-Augmented Generation (RAG) is a sophisticated architecture that combines retrieval-based methods and generative AI to enhance the quality and relevance of information generated by AI models. Specifically, GraphRAG utilizes graph-based methods to further refine this process, providing a structured approach to managing and utilizing complex data relationships. This report delves into the specifics of GraphRAG, highlighting its technical components, integration with related technologies, applications, benefits, and the challenges it faces.

## Technical Details

GraphRAG is an extension of the conventional RAG model, incorporating graph-based database systems to exploit structured data relationships:

- **Structure and Components**: GraphRAG employs a dual-system workflow combining a retrieval modu

In [19]:
print(final_report)

The research report on diversified mid-cap stocks within the Sensex universe has been synthesized and verified for factual accuracy. Below is the final version of the report incorporating critiques and improvements:

---

# Research Report: Top 10 Diversified Mid-Cap Stocks to Invest in Within the Sensex Universe

## Executive Summary
This report examines investment opportunities in mid-cap stocks within the Sensex universe, offering insights into trends and stock recommendations. It highlights 10 promising mid-cap stocks, considering recent market conditions, and provides comprehensive analysis to support investment decisions.

## Introduction
Mid-cap stocks represent a balanced investment opportunity, offering higher growth potential than large-caps and less volatility compared to small-caps. This report aims to identify the top 10 diversified mid-cap stocks that present viable investment opportunities today, backed by recent research, market performance, and trends in the economic l

In [17]:
print(final_report)

The team has successfully synthesized, fact-checked, and critiqued a comprehensive research report on the best midcap Sensex stocks to invest in today. Here is a summary of the key findings and recommendations:

### Key Insights from the Report

1. **Expert Recommendations**: Analysts at reputable firms like Sharekhan and Axis Direct recommend specific midcap stocks due to their growth potential. Godrej Consumer and Hero MotoCorp are among the highlighted short-term opportunities.

2. **Market Trends**:
   - **Technology and IT**: High interest in digital transformation places IT-focused midcaps in a favorable position.
   - **Healthcare**: Continued growth in the healthcare sector benefits midcap companies with robust R&D capabilities.
   - Broader market trends show that sectors leveraging government incentives and sustainability practices are poised for potential growth.

3. **Investment Considerations**:
   - **Risks and Volatility**: Midcap stocks, while offering growth potential,

In [23]:
import re
import openai
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.csv import CSVKnowledgeBase
# from agno.vectordb.pgvector import PgVector
from agno.tools.duckduckgo import DuckDuckGoTools

# ---------------------------
# Set up CSV Knowledge Base (Internal Knowledge)
# ---------------------------
# vector_db = PgVector(
#     table_name="csv_documents",
#     db_url="postgresql+psycopg://ai:ai@localhost:5532/ai",  # Update credentials as needed
# )
# csv_kb = CSVKnowledgeBase(
#     path="path/to/your/data.csv",  # Update with your CSV file path
#     vector_db=vector_db,
# )
# csv_kb.load(recreate=False)

# ---------------------------
# Flag to enable/disable internal resource usage
# ---------------------------
use_internal = False  # Set to False to disable internal research

# ---------------------------
# Specialized Agents for the Research Team
# ---------------------------
query_refiner_agent = Agent(
    name="QueryRefinerAgent",
    role="Refine and break query into multiple steps to be executed.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Break the query into simpler sub-questions if needed."],
    show_tool_calls=False,
    markdown=True,
)

class InternalResearchAgent(Agent):
    def probe(self, query):
        result = self.run(query).content.strip()
        return len(result) > 50

internal_research_agent = InternalResearchAgent(
    name="InternalResearchAgent",
    role="Search the CSV-based internal knowledge for relevant information.",
    model=OpenAIChat(id="gpt-4o"),
    knowledge=csv_kb,
    instructions=["Retrieve relevant internal information from the CSV knowledge base."],
    show_tool_calls=False,
    markdown=True,
)

external_research_agent = Agent(
    name="ExternalResearchAgent",
    role="Fetch up-to-date information from the web.",
    model=OpenAIChat(id="gpt-4o"),
    tools=[DuckDuckGoTools()],
    instructions=["Perform a web search and include sources in your response."],
    show_tool_calls=True,
    markdown=True,
)

summarization_agent = Agent(
    name="SummarizationAgent",
    role="Summarize lengthy text into concise insights.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Provide a concise summary of the provided text."],
    show_tool_calls=False,
    markdown=True,
)

citation_agent = Agent(
    name="CitationAgent",
    role="Extract and format citations from text.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Extract cited sources, URLs, or references from the text."],
    show_tool_calls=False,
    markdown=True,
)

trend_agent = Agent(
    name="TrendAnalysisAgent",
    role="Identify recent trends related to the query.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Identify any recent trends or developments relevant to the query."],
    show_tool_calls=False,
    markdown=True,
)

# ---------------------------
# Specialized Agents for the Synthesis Team
# ---------------------------
synthesis_agent = Agent(
    name="SynthesisAgent",
    role="Synthesize a comprehensive research report from provided inputs.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=[
        "Using the following inputs, generate a detailed and well-organized research report.",
        "Inputs include: Original Query, Refined Query, Research Plan, Internal Summary, External Summary, Trends, and Citations.",
        "Be sure to incorporate a clear chain-of-thought reasoning that explains how you integrated the inputs."
    ],
    show_tool_calls=False,
    markdown=True,
)

fact_checker_agent = Agent(
    name="FactCheckerAgent",
    role="Verify the factual accuracy of the report.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Check the report for factual errors and inconsistencies."],
    show_tool_calls=False,
    markdown=True,
)

critique_agent = Agent(
    name="CritiqueAgent",
    role="Critique the final report and suggest improvements.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=["Provide constructive feedback and suggestions to improve clarity and depth."],
    show_tool_calls=False,
    markdown=True,
)

# ---------------------------
# Decision Agent with Probe Approach
# ---------------------------
class DecisionAgentWithProbe:
    def __init__(self, internal_agent):
        self.internal_agent = internal_agent
        self.agent = Agent(
            name="DecisionAgent",
            role="Decide which sub-agents to invoke based on the query and internal knowledge relevance.",
            model=OpenAIChat(id="gpt-4o"),
            instructions=[
                "You are a strategic decision-maker for a multi-agent research system.",
                "Based on the research query and a hint about internal knowledge relevance, decide which of the following agents to invoke:",
                "InternalResearchAgent, ExternalResearchAgent, QueryRefinerAgent, SummarizationAgent,",
                "CitationAgent, TrendAnalysisAgent, FactCheckerAgent, CritiqueAgent.",
                "Return a comma-separated list of agent names in lowercase."
            ],
            show_tool_calls=False,
            markdown=True,
        )
    
    def decide(self, query):
        if use_internal:
            internal_relevant = self.internal_agent.probe(query)
        else:
            internal_relevant = False
        hint = "internal relevant" if internal_relevant else "internal not relevant"
        decision_prompt = f"{query}\nHint: {hint}"
        response = self.agent.run(decision_prompt)
        decision_text = response.content.strip()
        decisions = [agent.strip().lower() for agent in decision_text.split(',')]
        return decisions

decision_agent = DecisionAgentWithProbe(internal_research_agent)

# ---------------------------
# Improved Tournament Evaluator Agent
# ---------------------------
# class TournamentEvaluatorAgent:
#     def evaluate(self, candidates, category, instructions=None, max_tokens=100):
#         if instructions is None:
#             instructions = (
#                 f"Rank the following candidate responses for '{category}' by quality, clarity, and relevance. "
#                 "Return only the candidate number (1-indexed) of the best candidate."
#             )
#         prompt = instructions + "\n\n"
#         for idx, cand in enumerate(candidates):
#             prompt += f"Candidate {idx+1}:\n{cand}\n\n"
#         prompt += "Which candidate is best? Output just the candidate number."
#         response = openai.ChatCompletion.create(
#             model="gpt-4o",
#             messages=[
#                 {"role": "system", "content": "You are an impartial evaluator."},
#                 {"role": "user", "content": prompt}
#             ],
#             max_tokens=max_tokens
#         )
#         eval_result = response["choices"][0]["message"]["content"].strip()
#         try:
#             best_index = int(eval_result) - 1
#             if best_index < 0 or best_index >= len(candidates):
#                 best_index = 0
#         except Exception:
#             best_index = 0
#         return candidates[best_index]

# ---------------------------
# Define Agent Teams with Mandatory Planning and Reasoning
# ---------------------------
# Research Team: Must plan the research process as the first step.
research_team = Team(
    mode="coordinate",
    members=[
        query_refiner_agent,
        external_research_agent,
        summarization_agent,
        citation_agent,
        trend_agent
        # Note: We omit internal research here if use_internal is False.
    ],
    model=OpenAIChat(id="gpt-4o"),
    success_criteria="Detailed research plan and comprehensive research inputs, including summaries, citations, and trend analysis.",
    instructions=[
        "First, produce a detailed research plan based on the refined query.",
        "Then, gather and summarize research inputs from the web (and internal data if enabled).",
        "Include sources and relevant citations."
    ],
    show_tool_calls=False,
    markdown=True,
)

# Synthesis Team: Must include mandatory reasoning in the final synthesis.
synthesis_team = Team(
    mode="coordinate",
    members=[
        synthesis_agent,
        fact_checker_agent,
        critique_agent
    ],
    model=OpenAIChat(id="gpt-4o"),
    success_criteria="A final deep research report with clear reasoning, verified facts, and constructive critique.",
    instructions=[
        "Using the research inputs, generate a comprehensive research report.",
        "Include a detailed chain-of-thought reasoning explaining how the inputs were integrated and why they were prioritized.",
        "Then, fact-check and critique the final report for improvements."
    ],
    show_tool_calls=False,
    markdown=True,
)

# ---------------------------
# Main Reasoning Agent with Hybrid Approach
# ---------------------------
class MainReasoningAgentSelective:
    def __init__(self, decision_agent, research_team, synthesis_team, query_refiner):
        self.decision_agent = decision_agent
        self.research_team = research_team
        self.synthesis_team = synthesis_team
        self.query_refiner = query_refiner
        # self.evaluator = TournamentEvaluatorAgent()

    def run(self, query):
        # Step 0: Decide which sub-agents to invoke.
        decisions = self.decision_agent.decide(query)
        print("[DecisionAgent] Selected agents:", decisions)
        
        # Step 1: Optionally refine the query.
        if "queryrefineragent" in decisions:
            refined = self.query_refiner.run(query).content.strip()
        else:
            refined = query
        print("[Main] Refined Query:", refined)
        
        # Step 2: Run the research team.
        # If internal research is enabled and selected, we could merge its results; for simplicity, we use external only if disabled.
        research_input = f"Research query: {query}\nRefined query: {refined}"
        research_response = self.research_team.run(research_input)
        research_summary = research_response.content.strip()
        
        
        # Step 3: Use the synthesis team to produce the final report.
        synthesis_input = (
            f"Original Query: {query}\n"
            f"Refined Query: {refined}\n\n"
            f"Research Summary: {research_summary}\n"
        )
        print("synthesis_input :",synthesis_input)
        synthesis_response = self.synthesis_team.run(synthesis_input)
        report = synthesis_response.content.strip()
        print("[Main] Final Synthesis Completed.")
        
        return report

# ---------------------------
# Main Execution
# ---------------------------
# if __name__ == "__main__":
#     openai.api_key = "YOUR_OPENAI_API_KEY"
    
main_agent = MainReasoningAgentSelective(
    decision_agent=decision_agent,
    research_team=research_team,
    synthesis_team=synthesis_team,
    query_refiner=query_refiner_agent,
)
    


In [24]:
research_query = "Best 10 stocks to invest today in midcap sensex."
final_report = main_agent.run(research_query)

print("\n=== Final Deep Research Report ===\n")
print(final_report)


[DecisionAgent] Selected agents: ['externalresearchagent', 'trendanalysisagent', 'factcheckeragent', 'criticiqueagent']
[Main] Refined Query: Best 10 stocks to invest today in midcap sensex.
synthesis_input : Original Query: Best 10 stocks to invest today in midcap sensex.
Refined Query: Best 10 stocks to invest today in midcap sensex.

Research Summary: ### Research Report: Best Midcap Stocks to Invest in Today

#### Overview

**Objective:** Identify the best 10 midcap stocks for investment today in the Sensex, including a detailed analysis, recent trends, and expert recommendations to form comprehensive insights into the midcap sector.

---

#### Recommended Midcap Stocks

Based on available analyses and expert opinions as of October 2023:

1. **Howard Hughes Corporation (NYSE: HHH)**
   - Focus: Real estate development and management
   - Analysis: Strong market positioning with growth potential.
   - [Read more](https://www.nasdaq.com/articles/5-top-mid-cap-stocks-to-buy-now-accord

In [25]:
print(final_report)


The comprehensive research report on the best 10 midcap stocks to invest in today within the Sensex has been successfully synthesized, fact-checked, and critiqued.

### Summary of Key Findings:

- **Objective & Methodology**: The report aims to identify and analyze midcap stocks with strong growth potential, supported by expert evaluations and market trends.
  
- **Recommended Stocks**: The report provides a detailed analysis of 10 midcap stocks within the Sensex, considering factors such as industry trends, expert recommendations, and potential risks.

- **Broader Industry Insights**: It discusses macroeconomic conditions, policy reforms, and industry adaptations that influence the midcap sector.

- **Expert Recommendations**: A strategic investment approach is advised, recommending diversification to balance growth and risk.

### Critique Highlights:

- **Areas of Improvement**:
  - **Add an Executive Summary**: A brief section at the beginning can provide an overview of key findings

In [11]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools
from agno.team import Team

web_agent = Agent(
    name="Web Agent",
    role="Search the web for information",
    model=OpenAIChat(id="gpt-4o"),
    tools=[DuckDuckGoTools()],
    instructions="Always include sources",
    show_tool_calls=True,
    markdown=True,
)

finance_agent = Agent(
    name="Finance Agent",
    role="Get financial data",
    model=OpenAIChat(id="gpt-4o"),
    tools=[YFinanceTools(stock_price=True, analyst_recommendations=True, company_info=True)],
    instructions="Use tables to display data",
    show_tool_calls=True,
    markdown=True,
)

agent_team = Team(
    mode="coordinate",
    members=[web_agent, finance_agent],
    model=OpenAIChat(id="gpt-4o"),
    success_criteria="A comprehensive financial news report with clear sections and data-driven insights.",
    instructions=["Always include sources", "Use tables to display data"],
    show_tool_calls=True,
    markdown=True,
)

agent_team.print_response("What's the market outlook and financial performance of AI semiconductor companies?", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ What's the market outlook and financial performance of AI semiconductor companies?                              ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (32.8s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                  Market Outlook for AI Semiconductor Industry                                   ┃
┃                                                                                                                 ┃
┃  1 Growth Projections:                                                                                          ┃
┃     • The worldwide AI integrated circuit (IC) revenue is estimated to have reached $54 billion in 2023, with   ┃
┃       projections to rise to $71 billion in 2024 and reach $92 billion by 2025 (]8;id=676470;https://www.semiconductorintelligence.com/ai-semiconductor-market/\Source]8;;\).                        ┃
┃  2 Market Dynamics:                                                                                             ┃
┃     • After experiencing a downturn in the first half of 2023, the semiconductor industry is expected to        ┃
┃       recover through 2024. The global semiconductor market might surpass $1 trillion by 2030 (]8;id=762104;https://www.mckinsey.com/~/media/mckinsey/industries/semiconductors/our+insights/mckinsey+on+semiconductors+2024/mck_semiconductors_2024_webpdf.pdf\Source]8;;\).         ┃
┃  3 Challenges and Innovations:                                                                                  ┃
┃     • Automotive sectors signify a key growth driver, predicting growth from $76 billion in 2023 to $117        ┃
┃       billion over five years (]8;id=743217;https://www.pwc.com/gx/en/industries/technology/state-of-the-semicon-industry.html\Source]8;;\).                                                                         ┃
┃                                                                                                                 ┃
┃                           Financial Performance of Leading AI Semiconductor Companies                           ┃
┃                                                                                                                 ┃
┃                                                     NVIDIA                                                      ┃
┃                                                                                                                 ┃
┃  • Stock Performance: NVIDIA is experiencing its worst quarter since 2022 due to economic concerns. However,    ┃
┃    their performance remains robust amid challenges (]8;id=126529;https://www.msn.com/en-us/money/markets/why-nvidia-stock-is-on-track-to-have-its-worst-quarter-since-2022/ar-AA1BkCp8\Source]8;;\).                                                   ┃
┃  • Analysts' Views: Skepticism exists around growth, with potential drastic impacts if major customers withdraw ┃
┃    (]8;id=332867;https://www.msn.com/en-us/money/topstocks/why-one-analyst-and-nvidia-skeptic-says-the-chipmaker-might-not-grow-at-all-next-year/ar-AA1Bf4qU\Source]8;;\).                                                                                                    ┃
┃                                                                                                                 ┃
┃                                                      Intel      

In [8]:
Agent?

Init signature:
Agent(
    *,
    model: 'Optional[Model]' = None,
    name: 'Optional[str]' = None,
    agent_id: 'Optional[str]' = None,
    introduction: 'Optional[str]' = None,
    user_id: 'Optional[str]' = None,
    session_id: 'Optional[str]' = None,
    session_name: 'Optional[str]' = None,
    session_state: 'Optional[Dict[str, Any]]' = None,
    context: 'Optional[Dict[str, Any]]' = None,
    add_context: 'bool' = False,
    resolve_context: 'bool' = True,
    memory: 'Optional[AgentMemory]' = None,
    add_history_to_messages: 'bool' = False,
    num_history_responses: 'int' = 3,
    knowledge: 'Optional[AgentKnowledge]' = None,
    add_references: 'bool' = False,
    retriever: 'Optional[Callable[..., Optional[List[Dict]]]]' = None,
    references_format: "Literal['json', 'yaml']" = 'json',
    storage: 'Optional[Storage]' = None,
    extra_data: 'Optional[Dict[str, Any]]' = None,
    tools: 'Optional[List[Union[Toolkit, Callable, Function, Dict]]]' = None,
    show_tool_ca

In [ ]:

research_query = "Impact of renewable energy on global economies"
final_report = main_agent.run(research_query)

print("\n=== Final Deep Research Report ===\n")
print(final_report)


In [125]:
from sentence_transformers import util
import numpy as np
# embeddings = np.array(d['embedding'].to_pandas(), dtype=np.float32)
clusters = util.community_detection(d['embedding'].to_pandas(), min_community_size=3, threshold=0.7)

In [170]:
from openai import OpenAI
client = OpenAI()

query = 'Are there any patterns in playlist issues based on app version, country, or rating?'
query = 'tell me what people are saying about playlists'

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content" : "you and a sql design expert and can create complex queries to do analysis."
        },
        {
            "role": "user",
            "content": '''given this schema : 
ID: string
Source: string
CreatedAt: string
Language: string
Record_Sentiment: double
Tracked_Keywords: string (keywords present in the full issue)
Reasons: string   (short reason of having this issue)
Content: string   (main text having the full issue from user)
Summary: string   (summary of issue being faced)
metadata_AppID: string
metadata_ID: string
metadata_Language_Code: string
metadata_Language_Name: string
metadata_Reviewer: string
metadata_Score: double
metadata_Timestamp: double
metadata_Useful: double
metadata_Author: string
metadata_CountryISO2: string
metadata_CreatedAt: double
metadata_Rating: double
metadata_Upvotes: double
metadata_Version: string 
embedding: fixed_size_list<item: float>[384]
  child 0, item: float

----
ID: [["41a33eca-3cea-55bc-9eff-96cc5d8a881f"]]
Source: [["Playstore"]]
CreatedAt: [["2025-03-12T04:39:31Z"]]
Language: [["spa"]]
Record_Sentiment: [[null]]
Tracked_Keywords: [["Advertisements"]]
Reasons: [["Happy With The Service Of Spotify, Happy With No Ads"]]
Content: [["Excellent especially when there are no ads"]]
Summary: [["The user praised Spotify for being excellent, especially in the absence of ads."]]
metadata_AppID: [["com.spotify.music"]]

create SQL query for this natural language query : '''+query+'''
Table name is dataset, give me only the executeable query no reasoning or explainaition as an output. Think internally only.
give only string output without sql tag. 
'''
        }
    ]
)

print(completion.choices[0].message.content)

SELECT Content, Summary, Reasons, Tracked_Keywords FROM dataset WHERE Tracked_Keywords LIKE '%playlists%' OR Reasons LIKE '%playlists%' OR Content LIKE '%playlists%' OR Summary LIKE '%playlists%';


In [78]:
duckdb.query('''
SELECT 
    metadata_Version, 
    metadata_CountryISO2, 
    metadata_Rating, 
    COUNT(*) AS issue_count,
    AVG(metadata_Score) AS avg_score,
    AVG("Record Sentiment") AS avg_sentiment
FROM 
    dataset
WHERE 
    "Tracked Keywords" LIKE '%Advertisements%'
GROUP BY 
    metadata_Version, 
    metadata_CountryISO2, 
    metadata_Rating
ORDER BY 
    issue_count DESC;
''').to_df()

,metadata_Version,metadata_CountryISO2,metadata_Rating,issue_count,avg_score,avg_sentiment
0,9.0.18.604,None,NaN,1546,2.668823,NaN
1,9.0.20.604,None,NaN,1423,2.605762,NaN
2,None,None,NaN,1345,2.182156,NaN
3,9.0.22.543,None,NaN,786,2.482188,NaN
4,9.0.16.572,None,NaN,747,2.531459,NaN
...,...,...,...,...,...,...
688,8.9.64.548,None,NaN,1,5.000000,NaN
689,9.0.18,cz,4.0,1,NaN,NaN
690,9.0.20,gr,4.0,1,NaN,NaN
691,9.0.20,il,4.0,1,NaN,NaN


In [63]:
duckdb.query('''SELECT * FROM dataset WHERE "Content" LIKE '%Advertisements%' ''').to_df()

,ID,Source,CreatedAt,Language,Record Sentiment,Tracked Keywords,Reasons,Content,Summary,metadata_AppID,...,metadata_Reviewer,metadata_Score,metadata_Timestamp,metadata_Useful,metadata_Author,metadata_CountryISO2,metadata_CreatedAt,metadata_Rating,metadata_Upvotes,metadata_Version
0,e15810cc-bef5-534b-9f98-13d9cd49ca91,Appstore,2025-03-10T14:01:15Z,jpn,NaN,Advertisements,"Issue With Ad Length, Issue With Unskippable A...","Advertisements are too many, not funny, annoyi...",The user finds the ads on Spotify to be too ma...,spotify-music-and-podcasts,...,None,NaN,NaN,NaN,がたわはわだわまは、も,jp,1.741615e+09,3.0,0.0,9.0.24
1,7d7f38fe-470e-5621-a546-a1457356626b,Playstore,2025-03-10T13:15:40Z,eng,NaN,"Advertisements, Artists, Play, Playlist, Recom...","Issue With Playlist, Issue With Search Functio...",Only the same three songs play out of the whol...,The user experiences issues with the playlist ...,com.spotify.music,...,Lorelei Jasso,2.0,1.741613e+09,0.0,None,None,NaN,NaN,NaN,9.0.24.601
2,e7d3e9fb-fb77-58b6-8057-44d654063356,Playstore,2025-02-13T09:37:14Z,jpn,NaN,Advertisements,Issue With Ads After Every Song,Advertisements have really become visible. Ads...,The user finds Spotify's frequent ads after ev...,com.spotify.music,...,Dareios,2.0,1.739439e+09,2.0,None,None,NaN,NaN,NaN,9.0.16.572
3,48e48e4a-329f-54f0-8f07-bd8a3a2a8f83,Playstore,2025-02-13T08:56:49Z,deu,NaN,"Advertisements, Episode, Podcast, Premium Plan",Issue With Podcast Ads,Advertisements come during podcast episodes. W...,The user is frustrated with ads during podcast...,com.spotify.music,...,Juri Richter (Schwarzer Ritter),1.0,1.739437e+09,2.0,None,None,NaN,NaN,NaN,9.0.18.604
4,3c06f8f4-d979-50a9-b009-a3b204475ae2,Playstore,2025-02-13T00:16:26Z,tha,NaN,Advertisements,"Issue With Ads On Spotify, Issue With Frequent...",Advertisements are very frequent and abundant.,The user finds the advertisements on Spotify t...,com.spotify.music,...,ชัยชนะ พริ้งไสว,2.0,1.739406e+09,0.0,None,None,NaN,NaN,NaN,9.0.16.572
5,f8e8a9ae-51ee-5f29-ab0c-ec1f555c146e,Playstore,2025-02-10T15:03:39Z,vie,NaN,Advertisements,None,Advertisements for self-selected music,The user is suggesting Spotify should show few...,com.spotify.music,...,Anh khoa Tran,1.0,1.739200e+09,1.0,None,None,NaN,NaN,NaN,9.0.16.572
6,8222166d-e763-584a-9582-14957fd854e8,Playstore,2025-02-16T22:20:59Z,spa,NaN,"Advertisements, Download, Playlist, Search","Issue With Ads On Spotify, Issue With Excessiv...","Advertisements are like a scratched record, th...",The user finds Spotify's ads repetitive and fr...,com.spotify.music,...,Dogeloge,1.0,1.739744e+09,0.0,None,None,NaN,NaN,NaN,9.0.18.604
7,b24008eb-05a8-521d-9dd8-382273238ea8,Playstore,2025-02-15T11:34:01Z,pol,NaN,"Advertisements, Playlist, Premium Plan","Issue With Adding Songs To Playlist, Issue Wit...","okej okej, 1. Advertisements. 2. begging for p...",The user is frustrated with Spotify's advertis...,com.spotify.music,...,dotka mar,1.0,1.739619e+09,46.0,None,None,NaN,NaN,NaN,9.0.18.604
8,56d9a0b7-0968-5d47-bc21-4804e62ed774,Playstore,2025-02-15T10:49:42Z,ita,NaN,"Advertisements, Artists","Issue With Frequent Ads On Spotify, Happy With...",Advertisements are slightly too invasive but i...,The user finds Spotify's ads invasive in the f...,com.spotify.music,...,Andrea Parmeggiani,4.0,1.739617e+09,0.0,None,None,NaN,NaN,NaN,9.0.18.604
9,b74b8f20-cb12-5f4b-bf79-e6ef63e0236b,Appstore,2025-02-15T07:17:05Z,kor,NaN,"Advertisements, Play, Playlist",Issue With Playlist Not Playing,"Advertisements are really annoying, and even i...","The user finds advertisements annoying, experi...",spotify-music-and-podcasts,...,None,NaN,NaN,NaN,まさはなあやひたさはたはまさひ,kr,1.739604e+09,1.0,0.0,9.0.18


In [66]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-search-preview",
    web_search_options={},
    messages=[
        {
            "role": "user",
            "content": "What was a positive news story from today from india?",
        }
    ],
)

print(completion.choices[0].message.content)

Today, Indian and U.S. officials made significant progress toward a bilateral trade agreement aimed at reducing tariffs and easing non-tariff barriers. After several days of talks in New Delhi, both sides are optimistic about finalizing the first tranche of the deal by autumn. This development is expected to enhance economic cooperation and boost trade between the two nations. ([reuters.com](https://www.reuters.com/world/india-us-making-progress-towards-trade-deal-officials-say-2025-03-29/?utm_source=openai))


## India and U.S. Advance Toward Trade Agreement:
- [India and US making progress towards trade deal, officials say](https://www.reuters.com/world/india-us-making-progress-towards-trade-deal-officials-say-2025-03-29/?utm_source=openai) 


In [12]:
import asyncio

# Get the currently running event loop
loop = asyncio.get_running_loop()
print(loop)

<_UnixSelectorEventLoop running=True closed=False debug=False>


In [ ]:
from agno.knowledge.csv import CSVKnowledgeBase
from agno.vectordb.lancedb import LanceDb, SearchType

# Initialize LanceDB for the knowledge base
lancedb = LanceDb(
    uri="tmp/lancedb",  # your LanceDB storage location
    table_name="csv_documents",
    search_type=SearchType.hybrid,  # or choose your preferred search type
    embedder=OpenAIEmbedder(id="text-embedding-3-small")  # specify your embedder
)

# Create the CSV knowledge base using LanceDB
csv_kb = CSVKnowledgeBase(
    path="spotify_records.csv",  # update with your CSV file path
    vector_db=lancedb,
)

# Load the knowledge base
csv_kb.load(recreate=False)

In [3]:
# from agno.agent import Agent
# from agno.models.openai import OpenAIChat
# from agno.tools.duckduckgo import DuckDuckGoTools

# agent = Agent(
#     model=OpenAIChat(id="gpt-4o"),
#     tools=[DuckDuckGoTools()],
#     markdown=True
# )
# agent.print_response("What's happening in New York?", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ What's happening in New York?                                                                                   ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • duckduckgo_news(query=New York, max_results=5)                                                                ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (10.4s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Here's a snapshot of recent news happening in New York:                                                         ┃
┃                                                                                                                 ┃
┃ 🌆 ]8;id=196154;https://images2.minutemediacdn.com/image/upload/c_crop,w_5677,h_3193,x_0,y_102/c_fill,w_720,ar_16:9,f_auto,q_auto,g_auto/images/ImagnImages/mmsport/inside_the_mets/01jpzf808kjqv2weca07.jpg\Image]8;;\                                                                                                        ┃
┃ 🌆 ]8;id=950722;https://img-s-msn-com.akamaized.net/tenant/amp/entityid/AA1BoQsy.img?w=1024&h=576&m=4&q=81\Image]8;;\                                                                                                        ┃
┃ 🌆 ]8;id=943753;https://www.masslive.com/resizer/v2/PSPJUCNYQRBZXKVV43XBC23FNI.jpg?auth=a9e5e93941768bb0ceac5354e290615d76b018b92c7d7dd643a76d45dfce5ea4&width=1280&quality=90\Image]8;;\                                                                                                        ┃
┃ 🌆 ]8;id=454878;https://giantswire.usatoday.com/gcdn/authoring/authoring-images/2025/03/17/SGNT/82489767007-usatsi-25079640.jpg?crop=5999,3375,x0,y312&width=3200&height=1801&format=pjpg&auto=webp\Image]8;;\                                                                                                        ┃
┃  1 Former New York Mets Pitcher Finds New Opportunity with Yankees                                              ┃
┃     • A former Mets starter is making waves in spring training with the Yankees. ]8;id=16294;https://www.msn.com/en-us/sports/mlb/former-new-york-mets-starter-expected-to-make-yankees-rotation/ar-AA1BsqYQ\(Read more)]8;;\                    ┃
┃                                                                                                                 ┃
┃  2 Congestion Pricing in New York                                                                               ┃
┃     • Governor Kathy Hochul praised the success of New York's congestion pricing despite the federal government ┃
┃       extending the deadline to end the toll. ]8;id=652115;https://www.msn.com/en-us/news/us/new-york-gov-kathy-hochul-touts-congestion-pricing-success-after-feds-extend-deadline-to-end-toll/ar-AA1Bo9w8\(Read more)]8;;\                                                       ┃
┃                                                                                                                 ┃
┃  3 Charges Laid in Fatal New York Vacation Rental Fire                      

In [256]:
import json
import httpx

from agno.agent import Agent

def get_top_hackernews_stories(num_stories: int = 10) -> str:
    """
    Use this function to get top stories from Hacker News.

    Args:
        num_stories (int): Number of stories to return. Defaults to 10.

    Returns:
        str: JSON string of top stories.
    """

    # Fetch top story IDs
    response = httpx.get('https://hacker-news.firebaseio.com/v0/topstories.json')
    story_ids = response.json()

    # Fetch story details
    stories = []
    for story_id in story_ids[:num_stories]:
        story_response = httpx.get(f'https://hacker-news.firebaseio.com/v0/item/{story_id}.json')
        story = story_response.json()
        if "text" in story:
            story.pop("text", None)
        stories.append(story)
    return json.dumps(stories)


agent = Agent(tools=[get_top_hackernews_stories], show_tool_calls=True, markdown=True)
agent.print_response("Summarize the top 5 stories on hackernews?", stream=True)

▰▰▰▰▰▰▱ Thinking...
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Summarize the top 5 stories on hackernews?                                                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ • get_top_hackernews_stories(num_stories=5)                                                                     ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (11.9s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Here are the top 5 stories on Hacker News right now:                                                            ┃
┃                                                                                                                 ┃
┃  1 Public Secrets Exposure Leads to Supply Chain Attack on GitHub CodeQL                                        ┃
┃     • Author: cyberbender                                                                                       ┃
┃     • Score: 211 points                                                                                         ┃
┃     • Comments: 40                                                                                              ┃
┃     • Summary: A security incident involving the exposure of public secrets has led to a supply chain attack on ┃
┃       GitHub's CodeQL. The article provides details on how the incident occurred and its implications.          ┃
┃     • Link: ]8;id=274295;https://www.praetorian.com/blog/codeqleaked-public-secrets-exposure-leads-to-supply-chain-attack-on-github-codeql/\Read more]8;;\                                                                                           ┃
┃  2 Fragments of a Rare Merlin Manuscript from c. 1300                                                           ┃
┃     • Author: derbOac                                                                                           ┃
┃     • Score: 32 points                                                                                          ┃
┃     • Comments: 6                                                                                               ┃
┃     • Summary: Researchers have discovered fragments of a rare manuscript about Merlin from circa 1300 at       ┃
┃       Cambridge. The find sheds light on medieval literature and history.                                       ┃
┃     • Link: ]8;id=721967;https://www.cam.ac.uk/stories/merlin-manuscript-discovered-cambridge\Read more]8;;\                                                                                           ┃
┃  3 Blue95: A Desktop for Your Childhood Home's Computer Room                                                    ┃
┃     • Author: elvis70                                                                                           ┃
┃     • Score: 428 points                                                                                         ┃
┃     • Comments: 220                                                                                             ┃
┃     • Summary: The project "Blue95" ai

In [82]:
from openai import OpenAI
import json
client = OpenAI()

query = 'Are there any patterns in playlist issues based on app version, country, or rating?'
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are an AI analyzing customer issues."},
        {"role": "user", "content": f"Give me a formatted issues related to query: What issues do customers bring up when they talk about playlists? list and summarize each point be detailed with hierarchy of issues with 2 mentions each from: {aa}"},
    
    ]
)

print(completion.choices[0].message.content)

Based on customer feedback regarding playlists, several key issues emerge, which can be categorized into various hierarchical themes. These issues are summarized below:

1. **Ads and Promotions**
   - **Excessive Ads**: Many users complain about the overwhelming number of ads interrupting their listening experience, stating that ads frequently appear after every song or few songs, which becomes frustrating. ("Very bad, honestly not recommended. I try to listen to something in my playlist and boom, an ad for about 1 minute.")
   - **Encouragement to Upgrade to Premium**: Customers feel that the app aggressively prompts them to upgrade, creating a less enjoyable experience for free users by introducing multiple ads and limitations. ("You can't listen in sequence from the playlist, so what's the point of having a Playlist?!")

2. **Playlist Functionality**
   - **Shuffle Mode Limitation**: Users express dissatisfaction with the inability to play playlists in the desired order without havi